# Semi-Supervised Drone Annotation — Gesamtpipeline

Dieses Notebook integriert alle 18 Skripte (`main1.py` – `main18.py`) der Pipeline **eins zu eins**.
Der Code ist identisch mit den Skripten im Ordner [`scripts/`](../scripts/); jede Stufe wird hier
zusätzlich mit deutschen Erläuterungen dokumentiert.

## Übersicht der Pipeline

| Phase | Skripte | Beschreibung |
|---|---|---|
| **1. Datenaufbereitung** | main1–main4 | CVAT-XML prüfen, ins YOLO-Format konvertieren, Konsistenz prüfen, Train/Val-Split |
| **2. Baseline** | main5 | Training des Baseline-Modells (YOLOv8n, nur manuelle Labels) |
| **3. Detection-Pseudo-Labeling** | main6–main12 | Pseudo-Labels erzeugen, analysieren, filtern, balancieren, mergen, Fine-Tuning |
| **4. Tracking & GPS** | main13 | ByteTrack-Tracking mit GPS-Zuordnung (First-Seen-Report) |
| **5. Tracking-Pseudo-Labeling** | main14–main17 | Tracking-basierte Label-Propagation, Balancierung, Merge, Fine-Tuning |
| **6. Evaluation** | main18 | Vergleich aller drei Modelle auf demselben manuellen Validierungsset |

> **Hinweis:** Die Pfade in den Skripten verweisen auf die Original-Arbeitsumgebung
> (`C:\thesis\...` bzw. relative Pfade). Zum Ausführen müssen sie an die eigene Umgebung angepasst werden.


# Phase 1 — Datenaufbereitung (main1–main4)

## main1 — Validierung der Klassen-Zuordnung

Prüft **vor** der Konvertierung, ob alle Labels aus der CVAT-XML-Annotationsdatei in der `CLASS_MAP` enthalten sind. Fehlt ein Label, muss die `CLASS_MAP` zuerst aktualisiert werden — sonst gingen Annotationen bei der Konvertierung stillschweigend verloren.

In [ ]:
# ============================================================
# MAIN1 - VALIDIERUNG DER KLASSEN-ZUORDNUNG (CLASS_MAP vs. CVAT-XML)
# ============================================================
# Zweck:
# Prüft, ob alle Labels aus der CVAT-XML-Annotationsdatei in der
# CLASS_MAP enthalten sind, bevor die Konvertierung ins YOLO-Format
# gestartet wird. So werden fehlende oder ungenutzte Klassen früh erkannt.
# ============================================================

import xml.etree.ElementTree as ET

# Pfad zur CVAT-XML-Annotationsdatei
XML_PATH = "flight-mbg-v2-01-regionA/ann/video01_regionA.xml"

# Zuordnung: Klassenname -> YOLO-Klassen-ID
CLASS_MAP = {
    "pool": 0,
    "bottle": 1,
    "bucket": 2,
    "puddle": 3,
    "tire": 4,
    "watertank": 5,
    "dumpster": 6,
    "large_trash_bin": 7,
    "plastic_bag": 8,
    "small_trash_bin": 9,
    "storm_drain": 10,
}

tree = ET.parse(XML_PATH)
root = tree.getroot()

xml_labels = set()

# Alle Track-Labels aus der XML einsammeln und normalisieren
# (Kleinschreibung, Leerzeichen -> Unterstrich)
for track in root.findall("track"):
    label = track.get("label")
    if label:
        label_clean = label.strip().lower().replace(" ", "_")
        xml_labels.add(label_clean)

map_labels = set(CLASS_MAP.keys())

missing_in_map = sorted(xml_labels - map_labels)   # In XML vorhanden, aber nicht in CLASS_MAP
unused_in_map = sorted(map_labels - xml_labels)    # In CLASS_MAP vorhanden, aber nicht in XML

print("XML içindeki label'lar:")
for lbl in sorted(xml_labels):
    print("-", lbl)

print("\nCLASS_MAP içindeki label'lar:")
for lbl in sorted(map_labels):
    print("-", lbl)

print("\nXML'de var ama CLASS_MAP'te yok:")
print(missing_in_map if missing_in_map else "YOK")

print("\nCLASS_MAP'te var ama XML'de yok:")
print(unused_in_map if unused_in_map else "YOK")

# Ergebnis: CLASS_MAP ist nur dann ausreichend, wenn kein Label fehlt
if not missing_in_map:
    print("\n✅ CLASS_MAP bu XML için yeterli.")
else:
    print("\n⚠️ Yeni label bulundu, işlemden önce CLASS_MAP güncellenmeli.")

## main2 — Konvertierung CVAT-XML → YOLO-Format

Wandelt die CVAT-Annotationen in YOLO-Trainingslabels um. Besonderheit: Die Frame-Nummern auf der Festplatte und in der XML stimmen nicht überein, daher wird zunächst automatisch der beste **Offset** zwischen beiden Nummerierungen bestimmt. Anschließend werden die Bounding-Boxen in das normierte YOLO-Format (`class x_center y_center width height`) umgerechnet.

In [ ]:
# ============================================================
# MAIN2 - KONVERTIERUNG CVAT-XML -> YOLO-FORMAT (MIT FRAME-OFFSET)
# ============================================================
# Zweck:
# Wandelt die CVAT-XML-Annotationen in YOLO-Trainingslabels um.
# Da die Frame-Nummern auf der Festplatte und in der XML nicht
# identisch sind, wird zunächst automatisch der beste Offset
# zwischen beiden Nummerierungen bestimmt.
# ============================================================

import os
import cv2
import xml.etree.ElementTree as ET

DATASET_NAME = "video04_regionB"

# Eingabe: extrahierte Frames und CVAT-XML-Annotationen
FRAMES_DIR = "flight-mbg-v2-04-regionB/frames"
XML_PATH = f"flight-mbg-v2-04-regionB/ann/{DATASET_NAME}.xml"

# Ausgabe: YOLO-Dataset (Bilder + Labels)
OUT_IMG_DIR = f"datasets/{DATASET_NAME}/images"
OUT_LABEL_DIR = f"datasets/{DATASET_NAME}/labels"

os.makedirs(OUT_IMG_DIR, exist_ok=True)
os.makedirs(OUT_LABEL_DIR, exist_ok=True)

# Zuordnung: Klassenname -> YOLO-Klassen-ID
CLASS_MAP = {
    "pool": 0,
    "bottle": 1,
    "bucket": 2,
    "puddle": 3,
    "tire": 4,
    "watertank": 5,
    "dumpster": 6,
    "large_trash_bin": 7,
    "plastic_bag": 8,
    "small_trash_bin": 9,
    "storm_drain": 10,
}

def parse_frame_id(filename: str) -> int:
    # Extrahiert die Frame-Nummer aus dem Dateinamen (z. B. frame_000123.png -> 123)
    return int(os.path.splitext(filename)[0].split("_")[1])

def build_frame_dict(xml_path: str, class_map: dict):
    # Liest alle Bounding-Boxen aus der XML ein und gruppiert sie pro Frame
    tree = ET.parse(xml_path)
    root = tree.getroot()

    frame_dict = {}
    unknown_labels = set()

    for track in root.findall("track"):
        label = track.get("label")
        if not label:
            continue

        label_clean = label.strip().lower().replace(" ", "_")

        # Unbekannte Labels werden gesammelt und übersprungen
        if label_clean not in class_map:
            unknown_labels.add(label_clean)
            continue

        class_id = class_map[label_clean]

        for box in track.findall("box"):
            frame = int(box.get("frame"))
            xtl = float(box.get("xtl"))
            ytl = float(box.get("ytl"))
            xbr = float(box.get("xbr"))
            ybr = float(box.get("ybr"))

            frame_dict.setdefault(frame, []).append((class_id, xtl, ytl, xbr, ybr))

    return frame_dict, unknown_labels

def find_best_offset(disk_ids, xml_ids, step=24):
    # Bestimmt den besten Offset zwischen Disk-Frame-Nummern und XML-Frame-Nummern
    xml_set = set(xml_ids)

    # Kandidaten-Offsets nur als Differenz xml_frame - disk_frame erzeugen
    candidate_offsets = set()
    for d in disk_ids[:20]:
        for x in xml_ids[:500]:
            candidate_offsets.add(x - d)

    best_offset = None
    best_score = -1

    # Konsistenz des Musters nur über die ersten 20 Disk-Frames prüfen
    sample = disk_ids[:20]

    for offset in sorted(candidate_offsets):
        score = 0
        streak = 0

        for d in sample:
            if (d + offset) in xml_set:
                score += 1
                streak += 1
            else:
                streak = 0

        # Optional könnte man hier auch die Streak-Länge belohnen
        score2 = score

        if score2 > best_score:
            best_score = score2
            best_offset = offset

    return best_offset, best_score

frame_dict, unknown_labels = build_frame_dict(XML_PATH, CLASS_MAP)

print(f"✅ XML annotated unique frame sayısı: {len(frame_dict)}")
print(f"⚠️ Unknown labels: {unknown_labels}")

frame_files = sorted([f for f in os.listdir(FRAMES_DIR) if f.endswith('.png')])
disk_ids = [parse_frame_id(f) for f in frame_files]
xml_ids = sorted(frame_dict.keys())

print(f"✅ Diskteki toplam frame: {len(frame_files)}")

best_offset, best_score = find_best_offset(disk_ids, xml_ids, step=24)

print(f"✅ Seçilen offset: {best_offset}")
print(f"✅ İlk 20 frame üstünde eşleşme: {best_score}/20")

# Debug-Ausgabe: erste 10 Zuordnungen prüfen
print("\nİlk 10 eşleşme kontrolü:")
for d in disk_ids[:10]:
    print(f"local={d} -> xml={d + best_offset} -> {'OK' if (d + best_offset) in frame_dict else 'MISS'}")

written = 0
for filename in frame_files:
    local_frame_id = parse_frame_id(filename)
    xml_frame_id = local_frame_id + best_offset

    # Frames ohne Annotation überspringen
    if xml_frame_id not in frame_dict:
        continue

    img_path = os.path.join(FRAMES_DIR, filename)
    img = cv2.imread(img_path)
    if img is None:
        continue

    h, w, _ = img.shape

    cv2.imwrite(os.path.join(OUT_IMG_DIR, filename), img)

    # Bounding-Boxen ins normalisierte YOLO-Format umrechnen:
    # class x_center y_center width height (alle Werte relativ zur Bildgröße)
    label_path = os.path.join(OUT_LABEL_DIR, filename.replace(".png", ".txt"))
    with open(label_path, "w", encoding="utf-8") as f:
        for (class_id, xtl, ytl, xbr, ybr) in frame_dict[xml_frame_id]:
            x_center = ((xtl + xbr) / 2.0) / w
            y_center = ((ytl + ybr) / 2.0) / h
            width = (xbr - xtl) / w
            height = (ybr - ytl) / h
            f.write(f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}\n")

    written += 1

print(f"\n✅ Dataset oluşturuldu! ({written} image)")

## main3 — Konsistenzprüfung des Datasets

Einfache Qualitätskontrolle: Prüft für Train- und Val-Split, ob jedes Bild ein zugehöriges Label besitzt und umgekehrt. Unvollständige Paare würden das YOLO-Training verfälschen.

In [ ]:
# ============================================================
# MAIN3 - KONSISTENZPRÜFUNG DES DATASETS (IMAGE/LABEL-PAARE)
# ============================================================
# Zweck:
# Prüft für die Train- und Val-Splits, ob jedes Bild ein
# zugehöriges Label besitzt und umgekehrt. Fehlende Paare
# würden das YOLO-Training verfälschen.
# ============================================================

import os

dataset = "datasets/video01_regionA"

for split in ["train", "val"]:
    img_dir = os.path.join(dataset, split, "images")
    lbl_dir = os.path.join(dataset, split, "labels")

    # Basisnamen (ohne Endung) von Bildern und Labels sammeln
    imgs = sorted([os.path.splitext(f)[0] for f in os.listdir(img_dir) if f.endswith(".png")])
    lbls = sorted([os.path.splitext(f)[0] for f in os.listdir(lbl_dir) if f.endswith(".txt")])

    print(f"\n--- {split.upper()} ---")
    print("image:", len(imgs))
    print("label:", len(lbls))

    # Mengendifferenz: Bilder ohne Label bzw. Labels ohne Bild
    missing_lbl = sorted(set(imgs) - set(lbls))
    extra_lbl = sorted(set(lbls) - set(imgs))

    print("labeli olmayan image:", len(missing_lbl))
    print("image'i olmayan label:", len(extra_lbl))

## main4 — Train/Val-Split (80/20)

Teilt das konvertierte Dataset zufällig, aber reproduzierbar (Seed 42) in 80 % Trainings- und 20 % Validierungsdaten. Bilder ohne Label werden übersprungen.

In [ ]:
# ============================================================
# MAIN4 - TRAIN/VAL-SPLIT (80/20)
# ============================================================
# Zweck:
# Teilt das konvertierte Dataset zufällig, aber reproduzierbar
# (fester Seed) in 80 % Trainings- und 20 % Validierungsdaten auf.
# Bilder ohne zugehöriges Label werden übersprungen.
# ============================================================

import os
import shutil
import random

DATASET_NAME = "video04_regionB"

BASE = f"datasets/{DATASET_NAME}"
IMG_DIR = f"{BASE}/images"
LBL_DIR = f"{BASE}/labels"

TRAIN_IMG = f"{BASE}/train/images"
TRAIN_LBL = f"{BASE}/train/labels"

VAL_IMG = f"{BASE}/val/images"
VAL_LBL = f"{BASE}/val/labels"

# Zielordner anlegen
for d in [TRAIN_IMG, TRAIN_LBL, VAL_IMG, VAL_LBL]:
    os.makedirs(d, exist_ok=True)

# Fester Seed für Reproduzierbarkeit
random.seed(42)

# Sowohl PNG als auch JPG werden unterstützt
images = [f for f in os.listdir(IMG_DIR) if f.endswith((".png", ".jpg", ".jpeg"))]

print(f"Toplam image: {len(images)}")

random.shuffle(images)

# 80/20-Aufteilung
split = int(len(images) * 0.8)

train_files = images[:split]
val_files = images[split:]

def copy_files(files, img_out, lbl_out):
    count = 0

    for f in files:
        img_src = os.path.join(IMG_DIR, f)
        lbl_name = os.path.splitext(f)[0] + ".txt"
        lbl_src = os.path.join(LBL_DIR, lbl_name)

        # Bilder ohne Label werden übersprungen
        if not os.path.exists(lbl_src):
            print(f"⚠️ Label yok: {lbl_name}")
            continue

        shutil.copy(img_src, os.path.join(img_out, f))
        shutil.copy(lbl_src, os.path.join(lbl_out, lbl_name))

        count += 1

    return count

train_count = copy_files(train_files, TRAIN_IMG, TRAIN_LBL)
val_count = copy_files(val_files, VAL_IMG, VAL_LBL)

print(f"\n✅ Train: {train_count}")
print(f"✅ Val: {val_count}")
print("🔥 Split tamam!")

# Phase 2 — Baseline-Training (main5)

## main5 — Training des Baseline-Modells

Trainiert YOLOv8n ausschließlich auf den manuell annotierten Daten. Die hohe Auflösung (`imgsz=1024`) ist entscheidend für kleine Objekte aus der Drohnenperspektive; AdamW und Cosine-LR sorgen für ein stabiles Training. Dieses Modell ist die Referenz für alle semi-supervised Experimente.

In [ ]:
# ============================================================
# MAIN5 - TRAINING DES BASELINE-MODELLS (YOLOv8n)
# ============================================================
# Zweck:
# Trainiert das Baseline-Modell ausschließlich auf den manuell
# annotierten Daten. Die hohe Bildauflösung (imgsz=1024) ist
# wichtig für die Erkennung kleiner Objekte aus Drohnenperspektive.
# ============================================================

from ultralytics import YOLO

def main():
    # Vortrainiertes YOLOv8n-Modell als Startpunkt
    model = YOLO("yolov8n.pt")

    # Frühere, einfachere Trainingskonfiguration (zum Vergleich behalten):
    # model.train(
    #     data="data_main.yaml",
    #     epochs=100,
    #     imgsz=640,
    #     device=0,
    #     patience=20,
    #     batch=16
    # )
    model.train(
    data="data_main.yaml",
    epochs=150,          # mehr Epochen für besseres Lernen
    imgsz=1024,          # hohe Auflösung — wichtig für kleine Objekte
    device=0,
    patience=30,         # Early Stopping nach 30 Epochen ohne Verbesserung
    batch=16,
    workers=4,
    optimizer="AdamW",   # stabilere Optimierung
    lr0=0.001,
    cos_lr=True,         # Cosine-Learning-Rate-Schedule
    augment=True
    )

if __name__ == "__main__":
    main()

# Phase 3 — Detection-basiertes Pseudo-Labeling (main6–main12)

## main6 — Gezielte Pseudo-Label-Generierung (Detection-basiert)

Wendet das Baseline-Modell auf **alle** Video-Frames an. Die Zielklassen werden automatisch anhand der manuellen Klassenverteilung gewählt: Klassen mit zu wenigen manuellen Beispielen (< 20) gehen in die manuelle Prüfung, bereits stark vertretene Klassen (> 1000) werden ausgeschlossen. Die Auswahl wird als `target_config.json` gespeichert und von den Folgeskripten wiederverwendet.

In [ ]:
# ============================================================
# MAIN6 - GEZIELTE PSEUDO-LABEL-GENERIERUNG (DETECTION-BASIERT)
# ============================================================
# Zweck:
# Wendet das Baseline-Modell auf alle Video-Frames an, um
# Pseudo-Labels zu erzeugen. Zielklassen werden automatisch
# anhand der manuellen Klassenverteilung ausgewählt:
# unterrepräsentierte Klassen werden bevorzugt, sehr seltene
# oder bereits ausreichend vertretene Klassen ausgeschlossen.
# ============================================================

import os
import cv2
import glob
import json
from collections import Counter
from ultralytics import YOLO

# =========================
# PFADE
# =========================
MODEL_PATH = r"C:\thesis\runs\detect\train\weights\best.pt"
MANUAL_DATASET_ROOT = r"C:\thesis\datasets\merged_dataset"

THESIS_ROOT = r"C:\thesis"
ALL_FRAMES_ROOT = r"C:\thesis\all_frames"
PSEUDO_OUTPUT_ROOT = r"C:\thesis\runs\pseudo"

model = YOLO(MODEL_PATH)

CLASS_NAMES = {
    0: "pool",
    1: "bottle",
    2: "bucket",
    3: "puddle",
    4: "tire",
    5: "watertank",
    6: "dumpster",
    7: "large trash bin",
    8: "plastic bag",
    9: "small trash bin",
    10: "storm drain",
}

# =========================
# EINSTELLUNGEN FÜR DIE AUTOMATISCHE KLASSENAUSWAHL
# =========================
# Klassen mit sehr wenigen manuellen Beispielen sind für Pseudo-Labeling riskant.
# Diese werden statt automatischem Pseudo-Labeling zur manuellen Prüfung markiert.
MIN_MANUAL_FOR_AUTO_PSEUDO = 20

# Oberhalb dieser Anzahl gilt eine Klasse als ausreichend vertreten.
# So wird z. B. "watertank" automatisch ausgeschlossen.
MAX_MANUAL_FOR_AUTO_PSEUDO = 1000

# Für Klassen ohne manuelle Beispiele keine Pseudo-Labels erzeugen.
ALLOW_ZERO_SHOT_PSEUDO = False


def read_manual_distribution():
    # Zählt die Bounding-Boxen pro Klasse im manuellen Dataset (train + val)
    counter = Counter()

    for split in ["train", "val"]:
        label_dir = os.path.join(MANUAL_DATASET_ROOT, split, "labels")

        if not os.path.exists(label_dir):
            print(f"⚠️ Manual label klasörü yok: {label_dir}")
            continue

        for file in os.listdir(label_dir):
            if not file.endswith(".txt"):
                continue

            path = os.path.join(label_dir, file)

            with open(path, "r", encoding="utf-8") as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) < 5:
                        continue

                    cls = int(parts[0])
                    counter[cls] += 1

    return counter


def decide_target_classes(manual_counter):
    # Teilt die Klassen anhand der manuellen Verteilung in drei Gruppen auf:
    # target (Pseudo-Labeling), excluded (ausgeschlossen), manual_review (manuell prüfen)
    target_classes = []
    excluded_classes = []
    manual_review_classes = []

    for cls, name in CLASS_NAMES.items():
        count = manual_counter.get(cls, 0)

        if count == 0 and not ALLOW_ZERO_SHOT_PSEUDO:
            excluded_classes.append(cls)
            continue

        if count < MIN_MANUAL_FOR_AUTO_PSEUDO:
            manual_review_classes.append(cls)
            continue

        if count > MAX_MANUAL_FOR_AUTO_PSEUDO:
            excluded_classes.append(cls)
            continue

        target_classes.append(cls)

    return target_classes, excluded_classes, manual_review_classes


def find_videos():
    # Sucht alle AVI-Videos nach dem Muster flight-mbg-v2-*/avi/*.avi
    pattern = os.path.join(THESIS_ROOT, "flight-mbg-v2-*", "avi", "*.avi")
    video_paths = sorted(glob.glob(pattern))

    videos = {}

    for path in video_paths:
        # Ordnername: flight-mbg-v2-01-regionA
        parent = os.path.basename(os.path.dirname(os.path.dirname(path)))

        # Videoname: video01_regionA
        video_name = os.path.splitext(os.path.basename(path))[0]

        videos[video_name] = path

    return videos


def extract_all_frames(video_name, video_path):
    # Extrahiert alle Frames eines Videos als JPEG.
    # Bereits extrahierte Videos werden übersprungen.
    output_dir = os.path.join(ALL_FRAMES_ROOT, video_name)
    os.makedirs(output_dir, exist_ok=True)

    existing_frames = [
        f for f in os.listdir(output_dir)
        if f.lower().endswith((".png", ".jpg", ".jpeg"))
    ]

    if len(existing_frames) > 0:
        print(f"✅ Frame zaten var, geçiliyor: {output_dir}")
        return output_dir, True

    if not os.path.exists(video_path):
        print(f"❌ Video dosyası bulunamadı: {video_path}")
        return output_dir, False

    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        print(f"❌ Video açılamadı: {video_path}")
        return output_dir, False

    frame_id = 0

    while True:
        ret, frame = cap.read()

        if not ret:
            break

        frame_name = f"frame_{frame_id:06d}.jpg"
        frame_path = os.path.join(output_dir, frame_name)

        cv2.imwrite(frame_path, frame, [cv2.IMWRITE_JPEG_QUALITY, 90])
        frame_id += 1

    cap.release()

    if frame_id == 0:
        print(f"❌ Frame çıkarılamadı: {video_path}")
        return output_dir, False

    print(f"✅ {frame_id} frame çıkarıldı: {output_dir}")
    return output_dir, True


def save_target_config(target_classes, excluded_classes, manual_review_classes, manual_counter):
    # Speichert die Klassenauswahl als JSON — wird von main7/main8/main10 weiterverwendet
    os.makedirs(PSEUDO_OUTPUT_ROOT, exist_ok=True)

    config = {
        "target_classes": target_classes,
        "excluded_classes": excluded_classes,
        "manual_review_classes": manual_review_classes,
        "manual_distribution": {str(k): v for k, v in manual_counter.items()},
        "class_names": {str(k): v for k, v in CLASS_NAMES.items()},
    }

    out_path = os.path.join(PSEUDO_OUTPUT_ROOT, "target_config.json")

    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(config, f, indent=4, ensure_ascii=False)

    print(f"\n✅ Target config kaydedildi: {out_path}")


def run_pseudo_prediction(video_name, frames_dir, target_classes):
    # Führt die YOLO-Prädiktion auf allen Frames aus und speichert
    # die Ergebnisse als TXT-Labels inkl. Confidence-Werten
    print(f"\n🚀 Pseudo-label başlıyor: {video_name}")

    results = model.predict(
        source=frames_dir,
        project=PSEUDO_OUTPUT_ROOT,
        name=video_name,
        save=False,
        save_txt=True,
        save_conf=True,
        conf=0.25,
        imgsz=1024,
        classes=target_classes,
        exist_ok=True,
        stream=True
    )

    # stream=True liefert einen Generator — Iteration stößt die Verarbeitung an
    for _ in results:
        pass

    print(f"✅ Pseudo-label tamamlandı: {video_name}")


def main():
    manual_counter = read_manual_distribution()

    print("\n📊 Manual dataset class dağılımı:")
    for cls in sorted(CLASS_NAMES.keys()):
        print(f"{cls:2d} | {CLASS_NAMES[cls]:20s}: {manual_counter.get(cls, 0)}")

    target_classes, excluded_classes, manual_review_classes = decide_target_classes(manual_counter)

    print("\n🎯 Otomatik seçilen pseudo target class'lar:")
    for cls in target_classes:
        print(f"{cls:2d} | {CLASS_NAMES[cls]}")

    print("\n🚫 Pseudo'dan çıkarılan class'lar:")
    for cls in excluded_classes:
        print(f"{cls:2d} | {CLASS_NAMES[cls]}")

    print("\n👀 Manuel kontrol önerilen çok az class'lar:")
    for cls in manual_review_classes:
        print(f"{cls:2d} | {CLASS_NAMES[cls]}")

    if not target_classes:
        print("\n❌ Target class bulunamadı. Ayarları kontrol et.")
        return

    save_target_config(target_classes, excluded_classes, manual_review_classes, manual_counter)

    videos = find_videos()

    print("\n🎬 Bulunan videolar:")
    for name, path in videos.items():
        print(f"{name}: {path}")

    if not videos:
        print("\n❌ Video bulunamadı.")
        return

    for video_name, video_path in videos.items():
        print("\n==============================")
        print(f"🎬 İşleniyor: {video_name}")
        print("==============================")

        frames_dir, ok = extract_all_frames(video_name, video_path)

        if ok:
            run_pseudo_prediction(video_name, frames_dir, target_classes)
        else:
            print(f"⏭️ Atlandı: {video_name}")

    print("\n🔥 Targeted pseudo-label üretimi tamamlandı!")


if __name__ == "__main__":
    main()

## main7 — Analyse der Pseudo-Labels & Threshold-Empfehlung

Analysiert die rohen Pseudo-Labels pro Video und Klasse (Anzahl, mittlere/min/max Confidence) und leitet daraus über eine Heuristik **klassenweise Confidence-Schwellenwerte** ab, die als `threshold_suggestions.json` für die Filterung in main8 gespeichert werden.

In [ ]:
# ============================================================
# MAIN7 - ANALYSE DER PSEUDO-LABELS UND THRESHOLD-EMPFEHLUNG
# ============================================================
# Zweck:
# Analysiert die in main6 erzeugten Pseudo-Labels pro Video und
# Klasse (Anzahl, mittlere/min/max Confidence) und leitet daraus
# klassenweise Confidence-Schwellenwerte für die Filterung ab.
# Die Empfehlungen werden als JSON für main8 gespeichert.
# ============================================================

import os
import json
from collections import Counter, defaultdict

PSEUDO_ROOT = r"C:\thesis\runs\pseudo"
CONFIG_PATH = os.path.join(PSEUDO_ROOT, "target_config.json")

DEFAULT_CLASS_NAMES = {
    0: "pool",
    1: "bottle",
    2: "bucket",
    3: "puddle",
    4: "tire",
    5: "watertank",
    6: "dumpster",
    7: "large trash bin",
    8: "plastic bag",
    9: "small trash bin",
    10: "storm drain",
}


def load_config():
    # Lädt die in main6 erzeugte target_config.json
    if not os.path.exists(CONFIG_PATH):
        print(f"❌ Config bulunamadı: {CONFIG_PATH}")
        print("Önce main6.py çalışmalı.")
        return None

    with open(CONFIG_PATH, "r", encoding="utf-8") as f:
        config = json.load(f)

    class_names = {
        int(k): v for k, v in config.get("class_names", {}).items()
    }

    if not class_names:
        class_names = DEFAULT_CLASS_NAMES

    target_classes = [int(x) for x in config.get("target_classes", [])]
    excluded_classes = [int(x) for x in config.get("excluded_classes", [])]
    manual_review_classes = [int(x) for x in config.get("manual_review_classes", [])]

    return {
        "class_names": class_names,
        "target_classes": target_classes,
        "excluded_classes": excluded_classes,
        "manual_review_classes": manual_review_classes,
    }


def suggest_threshold(avg_conf):
    # Heuristik: je höher die mittlere Confidence einer Klasse,
    # desto höher darf der Filter-Schwellenwert sein
    if avg_conf >= 0.80:
        return 0.75
    elif avg_conf >= 0.65:
        return 0.60
    elif avg_conf >= 0.50:
        return 0.55
    else:
        return 0.65


def main():
    config = load_config()
    if config is None:
        return

    CLASS_NAMES = config["class_names"]
    TARGET_CLASSES = set(config["target_classes"])
    EXCLUDED_CLASSES = set(config["excluded_classes"])
    MANUAL_REVIEW_CLASSES = set(config["manual_review_classes"])

    print("🔍 Pseudo-label kontrolü başlıyor...\n")

    print("🎯 Target classes:")
    for cls in sorted(TARGET_CLASSES):
        print(f"{cls:2d} | {CLASS_NAMES.get(cls, cls)}")

    print("\n🚫 Excluded classes:")
    for cls in sorted(EXCLUDED_CLASSES):
        print(f"{cls:2d} | {CLASS_NAMES.get(cls, cls)}")

    print("\n👀 Manual review classes:")
    for cls in sorted(MANUAL_REVIEW_CLASSES):
        print(f"{cls:2d} | {CLASS_NAMES.get(cls, cls)}")

    total_counter = Counter()
    total_conf_values = defaultdict(list)
    unexpected_counter = Counter()

    video_names = [
        d for d in sorted(os.listdir(PSEUDO_ROOT))
        if os.path.isdir(os.path.join(PSEUDO_ROOT, d))
    ]

    for video_name in video_names:
        label_dir = os.path.join(PSEUDO_ROOT, video_name, "labels")

        if not os.path.exists(label_dir):
            continue

        txt_files = [f for f in os.listdir(label_dir) if f.endswith(".txt")]

        video_counter = Counter()
        video_conf_values = defaultdict(list)
        video_unexpected = Counter()

        frames_with_labels = 0
        total_boxes = 0

        for file in txt_files:
            file_has_label = False

            with open(os.path.join(label_dir, file), "r", encoding="utf-8") as f:
                for line in f:
                    parts = line.strip().split()

                    # YOLO-predict mit save_conf=True liefert:
                    # class x y w h conf
                    if len(parts) < 6:
                        continue

                    cls = int(parts[0])
                    conf = float(parts[5])

                    if cls in TARGET_CLASSES:
                        video_counter[cls] += 1
                        total_counter[cls] += 1

                        video_conf_values[cls].append(conf)
                        total_conf_values[cls].append(conf)

                        total_boxes += 1
                        file_has_label = True
                    else:
                        # Klassen außerhalb der Zielmenge separat zählen
                        video_unexpected[cls] += 1
                        unexpected_counter[cls] += 1

            if file_has_label:
                frames_with_labels += 1

        print("\n" + "=" * 60)
        print(f"📹 {video_name}")
        print("=" * 60)
        print(f"📄 Target label içeren frame: {frames_with_labels}")
        print(f"📦 Target bbox toplam: {total_boxes}")

        if total_boxes == 0:
            print("⚠️ Bu videoda target pseudo-label yok.")
            continue

        print("\n📊 Target class dağılımı:")

        for cls in sorted(video_counter.keys()):
            values = video_conf_values[cls]
            avg_conf = sum(values) / len(values)
            min_conf = min(values)
            max_conf = max(values)
            suggested = suggest_threshold(avg_conf)

            print(
                f"{cls:2d} | {CLASS_NAMES.get(cls, cls):20s} "
                f"| count: {video_counter[cls]:6d} "
                f"| avg: {avg_conf:.3f} "
                f"| min: {min_conf:.3f} "
                f"| max: {max_conf:.3f} "
                f"| önerilen th: {suggested:.2f}"
            )

        if video_unexpected:
            print("\n⚠️ Target dışı gelen class var:")
            for cls in sorted(video_unexpected.keys()):
                print(f"{cls:2d} | {CLASS_NAMES.get(cls, cls):20s}: {video_unexpected[cls]}")

    print("\n" + "=" * 70)
    print("📊 TOPLAM TARGET PSEUDO-LABEL DAĞILIMI")
    print("=" * 70)

    total_boxes_all = sum(total_counter.values())
    print(f"📦 Toplam target bbox: {total_boxes_all}\n")

    threshold_suggestions = {}

    for cls in sorted(total_counter.keys()):
        values = total_conf_values[cls]
        avg_conf = sum(values) / len(values)
        min_conf = min(values)
        max_conf = max(values)
        suggested = suggest_threshold(avg_conf)
        threshold_suggestions[cls] = suggested

        print(
            f"{cls:2d} | {CLASS_NAMES.get(cls, cls):20s} "
            f"| count: {total_counter[cls]:6d} "
            f"| avg: {avg_conf:.3f} "
            f"| min: {min_conf:.3f} "
            f"| max: {max_conf:.3f} "
            f"| önerilen th: {suggested:.2f}"
        )

    if unexpected_counter:
        print("\n⚠️ TOPLAM target dışı class:")
        for cls in sorted(unexpected_counter.keys()):
            print(f"{cls:2d} | {CLASS_NAMES.get(cls, cls):20s}: {unexpected_counter[cls]}")

    # Empfohlene Schwellenwerte für main8 speichern
    out_path = os.path.join(PSEUDO_ROOT, "threshold_suggestions.json")

    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(
            {str(k): v for k, v in threshold_suggestions.items()},
            f,
            indent=4,
            ensure_ascii=False
        )

    print(f"\n✅ Threshold önerileri kaydedildi: {out_path}")
    print("✅ Kontrol tamamlandı.")


if __name__ == "__main__":
    main()

## main8 — Filterung der Pseudo-Labels nach Confidence

Filtert die Pseudo-Labels anhand der klassenweisen Schwellenwerte aus main7 (Mindestschwelle 0.60). Boxen unterhalb des Schwellenwerts oder außerhalb der Zielklassen werden verworfen; die Confidence-Spalte wird entfernt, sodass gültige YOLO-Trainingslabels entstehen.

In [ ]:
# ============================================================
# MAIN8 - FILTERUNG DER PSEUDO-LABELS NACH CONFIDENCE
# ============================================================
# Zweck:
# Filtert die rohen Pseudo-Labels aus main6 anhand der in main7
# vorgeschlagenen klassenweisen Schwellenwerte. Boxen unterhalb
# des Schwellenwerts oder außerhalb der Zielklassen werden
# verworfen; die Confidence-Spalte wird für das YOLO-Trainings-
# format entfernt.
# ============================================================

import os
import json
from collections import Counter

PSEUDO_ROOT = r"C:\thesis\runs\pseudo"
INPUT_ROOT = r"C:\thesis\runs\pseudo"
OUTPUT_ROOT = r"C:\thesis\runs\pseudo_filtered"

CONFIG_PATH = os.path.join(PSEUDO_ROOT, "target_config.json")
THRESHOLD_PATH = os.path.join(PSEUDO_ROOT, "threshold_suggestions.json")

def load_json(path):
    if not os.path.exists(path):
        print(f"❌ Dosya bulunamadı: {path}")
        return None

    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def main():
    config = load_json(CONFIG_PATH)
    threshold_json = load_json(THRESHOLD_PATH)

    if config is None:
        print("Önce main6.py çalışmalı.")
        return

    if threshold_json is None:
        print("Önce main7.py çalışmalı.")
        return

    class_names = {int(k): v for k, v in config["class_names"].items()}
    target_classes = set(int(x) for x in config["target_classes"])

    thresholds = {
        int(k): float(v)
        for k, v in threshold_json.items()
    }

    # Sicherheits-Mindestschwelle:
    # main7 schlägt teils 0.55 vor — unter 0.60 lassen wir nicht zu.
    MIN_THRESHOLD = 0.60

    for cls in thresholds:
        thresholds[cls] = max(thresholds[cls], MIN_THRESHOLD)

    os.makedirs(OUTPUT_ROOT, exist_ok=True)

    print("🎯 Kullanılacak target class ve threshold değerleri:")
    for cls in sorted(target_classes):
        print(f"{cls:2d} | {class_names.get(cls, cls):20s} | th={thresholds.get(cls, MIN_THRESHOLD):.2f}")

    total_kept = Counter()
    total_removed = Counter()
    total_files_written = 0

    video_names = [
        d for d in sorted(os.listdir(INPUT_ROOT))
        if os.path.isdir(os.path.join(INPUT_ROOT, d))
    ]

    for video_name in video_names:
        input_label_dir = os.path.join(INPUT_ROOT, video_name, "labels")
        output_label_dir = os.path.join(OUTPUT_ROOT, video_name, "labels")

        if not os.path.exists(input_label_dir):
            continue

        os.makedirs(output_label_dir, exist_ok=True)

        kept = Counter()
        removed = Counter()
        files_written = 0
        empty_after_filter = 0

        for file in sorted(os.listdir(input_label_dir)):
            if not file.endswith(".txt"):
                continue

            in_path = os.path.join(input_label_dir, file)
            out_path = os.path.join(output_label_dir, file)

            new_lines = []

            with open(in_path, "r", encoding="utf-8") as f:
                for line in f:
                    parts = line.strip().split()

                    # Predict-Format:
                    # class x_center y_center width height conf
                    if len(parts) < 6:
                        continue

                    cls = int(parts[0])
                    conf = float(parts[5])

                    # Nicht-Zielklassen verwerfen
                    if cls not in target_classes:
                        removed[cls] += 1
                        total_removed[cls] += 1
                        continue

                    threshold = thresholds.get(cls, MIN_THRESHOLD)

                    if conf >= threshold:
                        # YOLO-Trainingsformat (ohne Confidence):
                        # class x_center y_center width height
                        new_lines.append(" ".join(parts[:5]) + "\n")
                        kept[cls] += 1
                        total_kept[cls] += 1
                    else:
                        removed[cls] += 1
                        total_removed[cls] += 1

            # Nur Dateien mit mindestens einer verbleibenden Box schreiben
            if new_lines:
                with open(out_path, "w", encoding="utf-8") as f:
                    f.writelines(new_lines)

                files_written += 1
                total_files_written += 1
            else:
                empty_after_filter += 1

        print("\n" + "=" * 60)
        print(f"📹 {video_name}")
        print("=" * 60)
        print(f"📄 Label kalan frame: {files_written}")
        print(f"🗑️ Boşa düşen frame: {empty_after_filter}")

        print("\n✅ Tutulan:")
        for cls in sorted(kept.keys()):
            print(f"{cls:2d} | {class_names.get(cls, cls):20s}: {kept[cls]}")

        print("\n❌ Silinen:")
        for cls in sorted(removed.keys()):
            print(f"{cls:2d} | {class_names.get(cls, cls):20s}: {removed[cls]}")

    print("\n" + "=" * 70)
    print("📊 TOPLAM FILTER SONUCU")
    print("=" * 70)

    print(f"📄 Toplam label kalan frame: {total_files_written}")

    print("\n✅ Toplam tutulan:")
    for cls in sorted(total_kept.keys()):
        print(f"{cls:2d} | {class_names.get(cls, cls):20s}: {total_kept[cls]}")

    print("\n❌ Toplam silinen:")
    for cls in sorted(total_removed.keys()):
        print(f"{cls:2d} | {class_names.get(cls, cls):20s}: {total_removed[cls]}")

    print("\n🔥 Filtering tamamlandı.")
    print(f"Output: {OUTPUT_ROOT}")

if __name__ == "__main__":
    main()

## main9 — Aufbau des Pseudo-Datasets

Kopiert zu jedem gefilterten Pseudo-Label das zugehörige Frame-Bild aus `all_frames` und baut so ein vollständiges Pseudo-Dataset (images + labels) pro Video auf.

In [ ]:
# ============================================================
# MAIN9 - AUFBAU DES PSEUDO-DATASETS (BILDER + LABELS KOPIEREN)
# ============================================================
# Zweck:
# Kopiert zu jedem gefilterten Pseudo-Label (aus main8) das
# zugehörige Frame-Bild aus all_frames und baut so ein
# vollständiges Pseudo-Dataset (images + labels) pro Video auf.
# ============================================================

import os
import shutil

# =========================
# EINGABE
# =========================
FILTERED_ROOT = r"C:\thesis\runs\pseudo_filtered"
ALL_FRAMES_ROOT = r"C:\thesis\all_frames"

# =========================
# AUSGABE
# =========================
PSEUDO_DATASET_ROOT = r"C:\thesis\datasets\pseudo_dataset"

IMAGE_EXTENSIONS = [".jpg", ".png", ".jpeg"]


def find_image(frame_dir, base_name):
    # Sucht das zugehörige Bild — probiert alle unterstützten Endungen durch
    for ext in IMAGE_EXTENSIONS:
        img_path = os.path.join(frame_dir, base_name + ext)
        if os.path.exists(img_path):
            return img_path, base_name + ext
    return None, None


def main():
    # Altes pseudo_dataset ggf. löschen (sauberer Neuaufbau)
    if os.path.exists(PSEUDO_DATASET_ROOT):
        shutil.rmtree(PSEUDO_DATASET_ROOT)

    video_names = [
        d for d in sorted(os.listdir(FILTERED_ROOT))
        if os.path.isdir(os.path.join(FILTERED_ROOT, d))
    ]

    total_images = 0
    total_labels = 0
    total_missing = 0

    for video_name in video_names:
        label_dir = os.path.join(FILTERED_ROOT, video_name, "labels")
        frame_dir = os.path.join(ALL_FRAMES_ROOT, video_name)

        out_img_dir = os.path.join(PSEUDO_DATASET_ROOT, video_name, "images")
        out_lbl_dir = os.path.join(PSEUDO_DATASET_ROOT, video_name, "labels")

        if not os.path.exists(label_dir):
            print(f"⏭️ Label klasörü yok, geçiliyor: {label_dir}")
            continue

        if not os.path.exists(frame_dir):
            print(f"⏭️ Frame klasörü yok, geçiliyor: {frame_dir}")
            continue

        os.makedirs(out_img_dir, exist_ok=True)
        os.makedirs(out_lbl_dir, exist_ok=True)

        copied_images = 0
        copied_labels = 0
        missing_images = 0

        for label_file in sorted(os.listdir(label_dir)):
            if not label_file.endswith(".txt"):
                continue

            base_name = os.path.splitext(label_file)[0]

            src_img, image_file = find_image(frame_dir, base_name)
            src_lbl = os.path.join(label_dir, label_file)

            # Label ohne zugehöriges Bild wird übersprungen und gezählt
            if src_img is None:
                print(f"⚠️ Görsel bulunamadı: {os.path.join(frame_dir, base_name)}(.jpg/.png/.jpeg)")
                missing_images += 1
                continue

            dst_img = os.path.join(out_img_dir, image_file)
            dst_lbl = os.path.join(out_lbl_dir, label_file)

            shutil.copy2(src_img, dst_img)
            shutil.copy2(src_lbl, dst_lbl)

            copied_images += 1
            copied_labels += 1

        total_images += copied_images
        total_labels += copied_labels
        total_missing += missing_images

        print(f"\n📹 {video_name}")
        print(f"✅ Kopyalanan image: {copied_images}")
        print(f"✅ Kopyalanan label: {copied_labels}")
        print(f"⚠️ Eksik image: {missing_images}")

    print("\n🔥 Pseudo dataset image+label kopyalama tamamlandı.")
    print(f"📦 Toplam image: {total_images}")
    print(f"📦 Toplam label: {total_labels}")
    print(f"⚠️ Toplam eksik image: {total_missing}")


if __name__ == "__main__":
    main()

## main10 — Klassenbalancierte Auswahl (Quota-Verfahren)

Wählt aus dem Pseudo-Dataset eine **klassenbalancierte** Teilmenge: Pro Klasse wird eine Quota berechnet (Zielwert 800 minus vorhandene manuelle Boxen, max. 1000 Pseudo-Boxen). Frames mit vielen Boxen unterrepräsentierter Klassen werden priorisiert, damit häufige Klassen das Training nicht dominieren.

In [ ]:
# ============================================================
# MAIN10 - KLASSENBALANCIERTE AUSWAHL DER PSEUDO-LABELS (QUOTA)
# ============================================================
# Zweck:
# Wählt aus dem Pseudo-Dataset (main9) eine klassenbalancierte
# Teilmenge für das Training aus. Pro Klasse wird eine Quota
# berechnet (Zielwert minus vorhandene manuelle Boxen), damit
# häufige Klassen das Training nicht dominieren.
# ============================================================

import os
import json
import shutil
import random
from collections import Counter, defaultdict

# =========================
# EINGABE / AUSGABE
# =========================
PSEUDO_DATASET_ROOT = r"C:\thesis\datasets\pseudo_dataset"
OUTPUT_ROOT = r"C:\thesis\datasets\pseudo_dataset_split"

CONFIG_PATH = r"C:\thesis\runs\pseudo\target_config.json"

TRAIN_IMG = os.path.join(OUTPUT_ROOT, "train", "images")
TRAIN_LBL = os.path.join(OUTPUT_ROOT, "train", "labels")

IMAGE_EXTS = (".jpg", ".jpeg", ".png")

# =========================
# EINSTELLUNGEN
# =========================
random.seed(42)

# Zielwert pro Klasse: manual count + pseudo count ≈ TARGET_TOTAL_PER_CLASS
TARGET_TOTAL_PER_CLASS = 800

# Maximale Anzahl an Pseudo-Boxen pro Klasse
MAX_PSEUDO_BBOX_PER_CLASS = 1000

# Obergrenze für die Gesamtzahl der Pseudo-Frames
MAX_TOTAL_FRAMES = 3000


def load_config():
    # Lädt Klassen, Zielklassen und manuelle Verteilung aus target_config.json
    if not os.path.exists(CONFIG_PATH):
        raise FileNotFoundError(f"Config bulunamadı: {CONFIG_PATH}. Önce main6.py çalışmalı.")

    with open(CONFIG_PATH, "r", encoding="utf-8") as f:
        config = json.load(f)

    class_names = {int(k): v for k, v in config["class_names"].items()}
    target_classes = [int(x) for x in config["target_classes"]]
    manual_distribution = {
        int(k): int(v)
        for k, v in config["manual_distribution"].items()
    }

    return class_names, target_classes, manual_distribution


def read_label_lines(label_path):
    # Liest YOLO-Labelzeilen und gibt (Klasse, bereinigte Zeile) zurück
    lines = []

    with open(label_path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()

            if len(parts) < 5:
                continue

            cls = int(parts[0])
            lines.append((cls, " ".join(parts[:5]) + "\n"))

    return lines


def find_all_pairs():
    # Sammelt alle Bild/Label-Paare aus allen Video-Unterordnern
    pairs = []

    video_names = [
        d for d in sorted(os.listdir(PSEUDO_DATASET_ROOT))
        if os.path.isdir(os.path.join(PSEUDO_DATASET_ROOT, d))
    ]

    for video_name in video_names:
        img_dir = os.path.join(PSEUDO_DATASET_ROOT, video_name, "images")
        lbl_dir = os.path.join(PSEUDO_DATASET_ROOT, video_name, "labels")

        if not os.path.exists(img_dir) or not os.path.exists(lbl_dir):
            print(f"⏭️ Eksik klasör, geçiliyor: {video_name}")
            continue

        images = [
            f for f in os.listdir(img_dir)
            if f.lower().endswith(IMAGE_EXTS)
        ]

        count = 0

        for img_file in images:
            base, ext = os.path.splitext(img_file)
            lbl_file = base + ".txt"

            src_img = os.path.join(img_dir, img_file)
            src_lbl = os.path.join(lbl_dir, lbl_file)

            if not os.path.exists(src_lbl):
                continue

            pairs.append({
                "video_name": video_name,
                "src_img": src_img,
                "src_lbl": src_lbl,
                "img_file": img_file,
                "base": base,
                "ext": ext,
            })

            count += 1

        print(f"📹 {video_name}: {count} pseudo image/label bulundu")

    return pairs


def main():
    class_names, target_classes, manual_distribution = load_config()
    target_classes = set(target_classes)

    # Alte Ausgabe wird gelöscht (sauberer Neuaufbau)
    if os.path.exists(OUTPUT_ROOT):
        shutil.rmtree(OUTPUT_ROOT)

    os.makedirs(TRAIN_IMG, exist_ok=True)
    os.makedirs(TRAIN_LBL, exist_ok=True)

    print("\n🎯 Target classes:")
    for cls in sorted(target_classes):
        print(f"{cls:2d} | {class_names.get(cls, cls)}")

    # =========================
    # 1) Pseudo-Quota pro Klasse berechnen
    # =========================
    quota = {}

    print("\n📊 Manual dağılıma göre pseudo quota:")

    for cls in sorted(target_classes):
        manual_count = manual_distribution.get(cls, 0)

        # Fehlende Boxen bis zum Zielwert, gedeckelt durch die Obergrenze
        missing = TARGET_TOTAL_PER_CLASS - manual_count
        pseudo_quota = max(0, min(missing, MAX_PSEUDO_BBOX_PER_CLASS))

        quota[cls] = pseudo_quota

        print(
            f"{cls:2d} | {class_names.get(cls, cls):20s} "
            f"| manual: {manual_count:5d} "
            f"| pseudo quota: {pseudo_quota:5d}"
        )

    if sum(quota.values()) == 0:
        print("\n⚠️ Hiç pseudo quota yok. Çıkılıyor.")
        return

    # =========================
    # 2) Kandidaten-Frames einlesen
    # =========================
    all_pairs = find_all_pairs()
    random.shuffle(all_pairs)

    print(f"\n📦 Toplam aday frame: {len(all_pairs)}")

    # Priorität: Frames mit vielen Boxen aus unterrepräsentierten Klassen zuerst
    def priority_score(item):
        label_lines = read_label_lines(item["src_lbl"])
        score = 0

        for cls, _line in label_lines:
            if cls in quota and quota[cls] > 0:
                score += quota[cls]

        return score

    all_pairs.sort(key=priority_score, reverse=True)

    # =========================
    # 3) Auswahl anhand der Bbox-Quota
    # =========================
    used_bbox_counter = Counter()
    selected_frames = 0
    skipped_frames = 0

    for item in all_pairs:
        if selected_frames >= MAX_TOTAL_FRAMES:
            break

        label_lines = read_label_lines(item["src_lbl"])

        kept_lines = []

        for cls, line in label_lines:
            if cls not in target_classes:
                continue

            # Klassen mit voller Quota nicht weiter aufnehmen
            if used_bbox_counter[cls] >= quota.get(cls, 0):
                continue

            kept_lines.append(line)
            used_bbox_counter[cls] += 1

        if not kept_lines:
            skipped_frames += 1
            continue

        video_name = item["video_name"]
        base = item["base"]
        ext = item["ext"]

        # Videoname als Präfix, um Namenskollisionen zu vermeiden
        new_img_name = f"{video_name}_{base}{ext}"
        new_lbl_name = f"{video_name}_{base}.txt"

        shutil.copy2(item["src_img"], os.path.join(TRAIN_IMG, new_img_name))

        with open(os.path.join(TRAIN_LBL, new_lbl_name), "w", encoding="utf-8") as f:
            f.writelines(kept_lines)

        selected_frames += 1

        # Abbruch, sobald alle Quotas erfüllt sind
        all_full = all(
            used_bbox_counter[cls] >= quota.get(cls, 0)
            for cls in target_classes
        )

        if all_full:
            break

    # =========================
    # 4) Bericht
    # =========================
    print("\n✅ Targeted balanced pseudo train oluşturuldu")
    print(f"📄 Seçilen frame: {selected_frames}")
    print(f"⏭️ Boş/işe yaramayan frame: {skipped_frames}")

    print("\n📊 Seçilen pseudo bbox dağılımı:")
    for cls in sorted(target_classes):
        print(
            f"{cls:2d} | {class_names.get(cls, cls):20s}: "
            f"{used_bbox_counter[cls]:5d} / quota {quota[cls]:5d}"
        )

    print(f"\n🔥 Output: {OUTPUT_ROOT}")


if __name__ == "__main__":
    main()

## main11 — Zusammenführung zum Semi-Supervised-Dataset

Kombiniert manuelles Training + balancierte Pseudo-Labels zu einem gemeinsamen Trainingsset. **Wichtig:** Die Validierung besteht ausschließlich aus manuellen Daten, damit die Evaluation nicht durch Pseudo-Labels verfälscht wird. Dateinamen erhalten Herkunfts-Präfixe (`manual_`/`pseudo_`).

In [ ]:
# ============================================================
# MAIN11 - ZUSAMMENFÜHRUNG ZUM SEMI-SUPERVISED-DATASET
# ============================================================
# Zweck:
# Kombiniert manuelle Trainingsdaten mit den balancierten
# Pseudo-Labels (main10) zu einem gemeinsamen Trainingsset.
# Wichtig: Die Validierung besteht ausschließlich aus manuellen
# Daten, damit die Evaluation nicht durch Pseudo-Labels
# verfälscht wird.
# ============================================================

import os
import shutil

MANUAL_ROOT = r"C:\thesis\datasets\merged_dataset"
PSEUDO_ROOT = r"C:\thesis\datasets\pseudo_dataset_split"
OUTPUT_ROOT = r"C:\thesis\datasets\semi_supervised_dataset"

IMAGE_EXTS = (".jpg", ".jpeg", ".png")


def clean_output():
    # Alte Ausgabe löschen und leere Ordnerstruktur anlegen
    if os.path.exists(OUTPUT_ROOT):
        shutil.rmtree(OUTPUT_ROOT)

    for split in ["train", "val"]:
        os.makedirs(os.path.join(OUTPUT_ROOT, split, "images"), exist_ok=True)
        os.makedirs(os.path.join(OUTPUT_ROOT, split, "labels"), exist_ok=True)


def copy_split(src_root, split, dst_split, prefix):
    # Kopiert einen Split und stellt jedem Dateinamen ein Präfix voran
    # ("manual" bzw. "pseudo"), um die Herkunft nachvollziehbar zu halten
    src_img = os.path.join(src_root, split, "images")
    src_lbl = os.path.join(src_root, split, "labels")

    dst_img = os.path.join(OUTPUT_ROOT, dst_split, "images")
    dst_lbl = os.path.join(OUTPUT_ROOT, dst_split, "labels")

    if not os.path.exists(src_img) or not os.path.exists(src_lbl):
        print(f"⏭️ Eksik klasör, geçiliyor: {src_root} / {split}")
        return 0

    count = 0

    for img_file in os.listdir(src_img):
        if not img_file.lower().endswith(IMAGE_EXTS):
            continue

        base, ext = os.path.splitext(img_file)
        lbl_file = base + ".txt"

        src_img_path = os.path.join(src_img, img_file)
        src_lbl_path = os.path.join(src_lbl, lbl_file)

        # Nur vollständige Bild/Label-Paare übernehmen
        if not os.path.exists(src_lbl_path):
            continue

        new_img_name = f"{prefix}_{base}{ext}"
        new_lbl_name = f"{prefix}_{base}.txt"

        shutil.copy2(src_img_path, os.path.join(dst_img, new_img_name))
        shutil.copy2(src_lbl_path, os.path.join(dst_lbl, new_lbl_name))

        count += 1

    print(f"✅ {prefix}: {split} → {dst_split}: {count} image-label kopyalandı")
    return count


def main():
    clean_output()

    # 1) Manuelles Training -> train
    manual_train = copy_split(
        src_root=MANUAL_ROOT,
        split="train",
        dst_split="train",
        prefix="manual"
    )

    # 2) Pseudo-Training -> train
    pseudo_train = copy_split(
        src_root=PSEUDO_ROOT,
        split="train",
        dst_split="train",
        prefix="pseudo"
    )

    # 3) Manuelle Validierung -> val (KEINE Pseudo-Labels in der Validierung!)
    manual_val = copy_split(
        src_root=MANUAL_ROOT,
        split="val",
        dst_split="val",
        prefix="manual"
    )

    print("\n🔥 Semi-supervised dataset hazır:")
    print(OUTPUT_ROOT)

    print("\n📊 Özet:")
    print(f"Train toplam: {manual_train + pseudo_train}")
    print(f"  - Manual train: {manual_train}")
    print(f"  - Pseudo train: {pseudo_train}")
    print(f"Val toplam: {manual_val}")
    print(f"  - Manual val only: {manual_val}")


if __name__ == "__main__":
    main()

## main12 — Fine-Tuning mit dem Semi-Supervised-Dataset

Feintuning des Baseline-Modells auf dem kombinierten Dataset (Detection-Pseudo-Variante). Kleinere Lernrate (`lr0=0.0003`) und weniger Epochen als beim Baseline-Training, da das Modell nur angepasst wird. Ergebnis: Modell `semi_iter1_targeted`.

In [ ]:
# ============================================================
# MAIN12 - FINE-TUNING MIT DEM SEMI-SUPERVISED-DATASET
# ============================================================
# Zweck:
# Feintuning des Baseline-Modells auf dem kombinierten Dataset
# (manuelle + Detection-Pseudo-Labels). Es wird bewusst eine
# kleinere Lernrate und weniger Epochen als beim Baseline-
# Training verwendet, da das Modell nur angepasst wird.
# ============================================================

from ultralytics import YOLO

def main():
    # Startpunkt: die besten Gewichte des Baseline-Trainings
    model = YOLO(r"C:\thesis\runs\detect\train\weights\best.pt")

    model.train(
        data=r"C:\thesis\data_semi.yaml",
        epochs=30,
        imgsz=1024,
        device=0,
        patience=7,
        batch=16,
        workers=4,
        optimizer="AdamW",
        lr0=0.0003,          # kleinere Lernrate fürs Fine-Tuning
        cos_lr=True,
        augment=True,
        name="semi_iter1_targeted"
    )

if __name__ == "__main__":
    main()

# Phase 4 — Tracking & GPS-Demo (main13)

## main13 — Objekt-Tracking mit GPS-Zuordnung (First-Seen-Report)

Anwendungsdemo: ByteTrack-Tracking auf einem Drohnenvideo. Für jede Track-ID wird nur der Zeitpunkt des **ersten Auftretens** in die CSV geschrieben — inklusive der über den Zeitstempel zugeordneten GPS-Position aus dem Fluglog und einem Google-Maps-Link. Zusätzlich entstehen ein annotiertes Video und eine Zusammenfassung. (Eine frühere Variante schrieb jede Detektion pro Frame; sie wurde durch diese First-Seen-Variante ersetzt.)

In [ ]:
# ============================================================
# MAIN13 - OBJEKT-TRACKING MIT GPS-ZUORDNUNG (FIRST-SEEN-REPORT)
# ============================================================
# Zweck:
# Führt ByteTrack-Tracking mit dem Baseline-Modell auf einem
# Drohnenvideo aus. Für jede Track-ID wird nur der Zeitpunkt
# des ERSTEN Auftretens in die CSV geschrieben, zusammen mit
# der zu diesem Frame passenden GPS-Position aus dem Fluglog
# (inkl. Google-Maps-Link). Zusätzlich werden ein annotiertes
# Video und eine Zusammenfassung erzeugt.
#
# Hinweis: Eine frühere Variante dieses Skripts schrieb jede
# Detektion pro Frame in die CSV (vollständiger Frame-Report);
# sie wurde durch diese First-Seen-Variante ersetzt.
# ============================================================

import os
import cv2
import csv
import pandas as pd
from collections import defaultdict
from ultralytics import YOLO

# =========================
# PFADE
# =========================
MODEL_PATH = r"C:\thesis\runs\detect\train\weights\best.pt"

VIDEO_PATH = r"C:\thesis\flight-mbg-v2-02-regionA\avi\video02_regionA.avi"
LOG_PATH = r"C:\thesis\flight-mbg-v2-02-regionA\log\video02_regionA.csv"

OUTPUT_ROOT = r"C:\thesis\runs\tracking_demo3"
RUN_NAME = "video02_regionA_tracking_first_seen_gps"

# =========================
# TRACKER-KONFIGURATION
# =========================
TRACKER_CFG = "bytetrack.yaml"
CONF_TH = 0.25
IMGSZ = 1024

# =========================
# KLASSENNAMEN
# =========================
CLASS_NAMES = {
    0: "pool",
    1: "bottle",
    2: "bucket",
    3: "puddle",
    4: "tire",
    5: "watertank",
    6: "dumpster",
    7: "large trash bin",
    8: "plastic bag",
    9: "small trash bin",
    10: "storm drain",
}

# =========================
# AUSGABEDATEIEN
# =========================
SAVE_DIR = os.path.join(OUTPUT_ROOT, RUN_NAME)
os.makedirs(SAVE_DIR, exist_ok=True)

OUTPUT_VIDEO = os.path.join(SAVE_DIR, "tracked_video.mp4")
OUTPUT_CSV = os.path.join(SAVE_DIR, "first_seen_tracking_report_with_gps.csv")
OUTPUT_SUMMARY = os.path.join(SAVE_DIR, "tracking_summary.txt")


def load_gps_log(log_path):
    # Lädt das GPS-Fluglog (CSV) und bereinigt es:
    # nur gültige Zeilen mit Zeitstempel, Breiten- und Längengrad
    if not os.path.exists(log_path):
        print(f"⚠️ GPS log bulunamadı: {log_path}")
        return None

    df = pd.read_csv(log_path)

    required_cols = ["latitude", "longitude", "time(millisecond)"]
    for col in required_cols:
        if col not in df.columns:
            print(f"⚠️ GPS log içinde kolon yok: {col}")
            return None

    df = df.dropna(subset=["latitude", "longitude", "time(millisecond)"]).copy()

    df["time(millisecond)"] = pd.to_numeric(df["time(millisecond)"], errors="coerce")
    df["latitude"] = pd.to_numeric(df["latitude"], errors="coerce")
    df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")

    df = df.dropna(subset=["latitude", "longitude", "time(millisecond)"])
    df = df.sort_values("time(millisecond)").reset_index(drop=True)

    print(f"✅ GPS log yüklendi: {len(df)} satır")
    return df


def get_nearest_gps(log_df, frame_idx, fps):
    """
    Ordnet die Video-Frame-Zeit der GPS-Log-Zeit zu.
    Der Zeitstempel der ersten Logzeile gilt als Videostart.
    """
    if log_df is None or len(log_df) == 0:
        return {
            "gps_time_ms": "",
            "datetime_utc": "",
            "datetime_local": "",
            "latitude": "",
            "longitude": "",
            "altitude": "",
            "google_maps_url": "",
        }

    log_start_ms = float(log_df["time(millisecond)"].iloc[0])
    frame_time_ms = log_start_ms + (frame_idx / fps) * 1000.0

    # Zeile mit der geringsten Zeitdifferenz zum Frame-Zeitpunkt suchen
    idx = (log_df["time(millisecond)"] - frame_time_ms).abs().idxmin()
    row = log_df.loc[idx]

    lat = float(row["latitude"])
    lon = float(row["longitude"])

    altitude = row["altitude(m)"] if "altitude(m)" in log_df.columns else ""
    datetime_utc = row["datetime(utc)"] if "datetime(utc)" in log_df.columns else ""
    datetime_local = row["datetime(local)"] if "datetime(local)" in log_df.columns else ""
    gps_time_ms = row["time(millisecond)"]

    google_maps_url = f"https://www.google.com/maps?q={lat},{lon}"

    return {
        "gps_time_ms": gps_time_ms,
        "datetime_utc": datetime_utc,
        "datetime_local": datetime_local,
        "latitude": lat,
        "longitude": lon,
        "altitude": altitude,
        "google_maps_url": google_maps_url,
    }


def main():
    model = YOLO(MODEL_PATH)
    gps_log = load_gps_log(LOG_PATH)

    # Video-Metadaten (FPS, Auflösung, Frame-Anzahl) auslesen
    cap = cv2.VideoCapture(VIDEO_PATH)
    if not cap.isOpened():
        print(f"❌ Video açılamadı: {VIDEO_PATH}")
        return

    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_video_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()

    print(f"🎥 Video FPS: {fps}")
    print(f"🎥 Video size: {width}x{height}")
    print(f"🎥 Toplam video frame: {total_video_frames}")

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, fps, (width, height))

    csv_file = open(OUTPUT_CSV, "w", newline="", encoding="utf-8")
    csv_writer = csv.writer(csv_file)

    # In die CSV wird pro Track-ID nur das erste Auftreten geschrieben
    csv_writer.writerow([
        "track_id",
        "first_seen_frame_idx",
        "class_id",
        "class_name",
        "first_seen_confidence",
        "x1", "y1", "x2", "y2",
        "center_x", "center_y",
        "bbox_width", "bbox_height",
        "gps_time_ms",
        "datetime_utc",
        "datetime_local",
        "drone_latitude",
        "drone_longitude",
        "altitude_m",
        "google_maps_url"
    ])

    total_detections = 0
    unique_tracks_all = set()
    unique_tracks_by_class = defaultdict(set)
    detections_per_class = defaultdict(int)

    # Dieses Set stellt sicher, dass jede track_id nur EINMAL in die CSV kommt
    written_track_ids = set()

    print("\n🚀 Tracking başlıyor...")

    results = model.track(
        source=VIDEO_PATH,
        tracker=TRACKER_CFG,
        conf=CONF_TH,
        imgsz=IMGSZ,
        persist=True,
        stream=True,
        save=False
    )

    frame_idx = 0

    for result in results:
        # Annotierten Frame ins Ausgabevideo schreiben
        plotted_frame = result.plot()
        writer.write(plotted_frame)

        boxes = result.boxes

        if boxes is not None and boxes.xyxy is not None:
            xyxy = boxes.xyxy.cpu().numpy()

            cls_list = boxes.cls.cpu().numpy().astype(int) if boxes.cls is not None else []
            conf_list = boxes.conf.cpu().numpy() if boxes.conf is not None else []

            # Ohne Track-ID liefert der Tracker -1; solche Boxen kommen nicht in den Report
            id_list = boxes.id.cpu().numpy().astype(int) if boxes.id is not None else [-1] * len(xyxy)

            for i in range(len(xyxy)):
                x1, y1, x2, y2 = xyxy[i]
                cls_id = int(cls_list[i]) if i < len(cls_list) else -1
                conf = float(conf_list[i]) if i < len(conf_list) else 0.0
                track_id = int(id_list[i]) if i < len(id_list) else -1

                class_name = CLASS_NAMES.get(cls_id, str(cls_id))

                total_detections += 1
                detections_per_class[class_name] += 1

                if track_id == -1:
                    continue

                unique_tracks_all.add(track_id)
                unique_tracks_by_class[class_name].add(track_id)

                # Bereits geschriebene Track-IDs überspringen (First-Seen-Prinzip)
                if track_id in written_track_ids:
                    continue

                written_track_ids.add(track_id)

                gps_info = get_nearest_gps(gps_log, frame_idx, fps)

                w = x2 - x1
                h = y2 - y1
                cx = x1 + w / 2
                cy = y1 + h / 2

                csv_writer.writerow([
                    track_id,
                    frame_idx,
                    cls_id,
                    class_name,
                    round(conf, 4),
                    round(float(x1), 2),
                    round(float(y1), 2),
                    round(float(x2), 2),
                    round(float(y2), 2),
                    round(float(cx), 2),
                    round(float(cy), 2),
                    round(float(w), 2),
                    round(float(h), 2),
                    gps_info["gps_time_ms"],
                    gps_info["datetime_utc"],
                    gps_info["datetime_local"],
                    gps_info["latitude"],
                    gps_info["longitude"],
                    gps_info["altitude"],
                    gps_info["google_maps_url"],
                ])

        if frame_idx % 100 == 0:
            print(f"🎞️ Frame işleniyor: {frame_idx}")

        frame_idx += 1

    writer.release()
    csv_file.close()

    # Zusammenfassung als Textdatei schreiben
    with open(OUTPUT_SUMMARY, "w", encoding="utf-8") as f:
        f.write("TRACKING SUMMARY WITH FIRST-SEEN GPS\n")
        f.write("============================\n")
        f.write(f"Video: {VIDEO_PATH}\n")
        f.write(f"GPS log: {LOG_PATH}\n")
        f.write(f"Model: {MODEL_PATH}\n")
        f.write(f"Tracker: {TRACKER_CFG}\n")
        f.write(f"FPS: {fps}\n")
        f.write(f"Video size: {width}x{height}\n")
        f.write(f"Processed frames: {frame_idx}\n")
        f.write(f"Total detections across frames: {total_detections}\n")
        f.write(f"Total unique tracked objects: {len(unique_tracks_all)}\n")
        f.write(f"Tracks written to CSV: {len(written_track_ids)}\n\n")

        f.write("Class-based summary:\n")
        for class_name in sorted(detections_per_class.keys()):
            f.write(
                f"- {class_name}: "
                f"detections={detections_per_class[class_name]}, "
                f"unique_tracks={len(unique_tracks_by_class[class_name])}\n"
            )

    print("\n✅ Tracking tamamlandı.")
    print(f"🎥 Video: {OUTPUT_VIDEO}")
    print(f"📄 First-seen CSV rapor: {OUTPUT_CSV}")
    print(f"📊 Özet rapor: {OUTPUT_SUMMARY}")


if __name__ == "__main__":
    main()

# Phase 5 — Tracking-basierte Label-Propagation (main14–main17)

## main14 — Tracking-basierte Label-Propagation

Kern der zweiten semi-supervised Methode: ByteTrack läuft auf allen Videos, und Pseudo-Labels werden anhand der **Track-Kontinuität** gefiltert — ein Track muss mindestens 3 Frames lang sein, eine mittlere Confidence ≥ 0.60 haben und zu ≥ 70 % klassenkonsistent sein. Frames aus dem manuellen Dataset werden ausgeschlossen (kein Val-Leakage). Umfangreiche CSV/JSON-Berichte dokumentieren die Filterung.

In [ ]:
# ============================================================
# MAIN14 - TRACKING-BASIERTE LABEL-PROPAGATION
# ============================================================
# Zweck:
# Führt das Baseline-YOLO-Modell mit ByteTrack auf allen Videos
# aus und erzeugt Pseudo-Labels anhand der Track-Kontinuität:
# Nur Tracks, die lang genug sind, eine ausreichende mittlere
# Confidence haben und klassenkonsistent sind, werden behalten.
#
# Ausgabe:
# C:\thesis\runs\tracking_pseudo\
#   video01_regionA\
#       images\
#       labels\
#   video02_regionA\
#       images\
#       labels\
#   tracking_pseudo_summary.csv
#   tracking_pseudo_detections.csv
# ============================================================

import os
import re
import cv2
import csv
import glob
import json
from collections import defaultdict, Counter
from ultralytics import YOLO


# =========================
# PFADE
# =========================
THESIS_ROOT = r"C:\thesis"

MODEL_PATH = r"C:\thesis\runs\detect\train\weights\best.pt"

# Falls main6.py bereits eine target_config.json erzeugt hat, wird sie genutzt.
# Andernfalls werden alle Klassen verwendet.
TARGET_CONFIG_PATH = r"C:\thesis\runs\pseudo\target_config.json"

OUTPUT_ROOT = r"C:\thesis\runs\tracking_pseudo"

# Bilder aus dem manuellen Dataset werden ausgeschlossen,
# um Val-/Test-Leakage zu verhindern.
MANUAL_DATASET_ROOT = r"C:\thesis\datasets\merged_dataset"


# =========================
# TRACKING-EINSTELLUNGEN
# =========================
TRACKER_CFG = "bytetrack.yaml"

# Detection-Schwellenwert für ByteTrack.
# Bewusst niedrig, damit der Tracker auch schwache Detektionen assoziieren kann.
TRACK_CONF_TH = 0.25

IMGSZ = 1024

# Mindest-Confidence, damit eine Box als Pseudo-Label akzeptiert wird.
KEEP_MIN_CONF = 0.60

# Mindestanzahl an Frames, in denen ein Track sichtbar sein muss.
MIN_TRACK_LENGTH = 3

# Schwellenwert für die mittlere Track-Confidence.
MIN_AVG_TRACK_CONF = 0.60

# Anteil der dominanten Klasse innerhalb eines Tracks.
# Beispiel: mindestens 70 % der Detektionen eines Tracks müssen dieselbe Klasse haben.
MIN_CLASS_CONSISTENCY = 0.70

# Minimale normierte Fläche, um winzige Boxen auszusortieren.
# 0.000001 = sehr klein, filtert nur völlig unsinnige Boxen heraus.
MIN_BOX_AREA_NORM = 0.000001

# Ausgabeformat der Bilder
OUTPUT_IMAGE_EXT = ".jpg"
JPEG_QUALITY = 90


# =========================
# KLASSENNAMEN
# =========================
DEFAULT_CLASS_NAMES = {
    0: "pool",
    1: "bottle",
    2: "bucket",
    3: "puddle",
    4: "tire",
    5: "watertank",
    6: "dumpster",
    7: "large trash bin",
    8: "plastic bag",
    9: "small trash bin",
    10: "storm drain",
}


def load_target_config():
    """
    Nutzt — falls vorhanden — die von main6.py erzeugte target_config.json:
    - target_classes
    - class_names

    Ohne Config werden alle Klassen als Ziel betrachtet.
    """
    if not os.path.exists(TARGET_CONFIG_PATH):
        print("⚠️ target_config.json bulunamadı. Tüm class'lar kullanılacak.")
        return DEFAULT_CLASS_NAMES, set(DEFAULT_CLASS_NAMES.keys())

    with open(TARGET_CONFIG_PATH, "r", encoding="utf-8") as f:
        config = json.load(f)

    class_names = {
        int(k): v
        for k, v in config.get("class_names", {}).items()
    }

    if not class_names:
        class_names = DEFAULT_CLASS_NAMES

    target_classes = set(int(x) for x in config.get("target_classes", []))

    if not target_classes:
        print("⚠️ target_config içinde target_classes boş. Tüm class'lar kullanılacak.")
        target_classes = set(class_names.keys())

    print("✅ target_config yüklendi.")
    print("🎯 Tracking pseudo target classes:")
    for cls in sorted(target_classes):
        print(f"{cls:2d} | {class_names.get(cls, cls)}")

    return class_names, target_classes


def find_videos():
    """
    Sucht die Videos unter C:\\thesis\\flight-mbg-v2-*\\avi\\*.avi.
    """
    pattern = os.path.join(THESIS_ROOT, "flight-mbg-v2-*", "avi", "*.avi")
    video_paths = sorted(glob.glob(pattern))

    videos = {}

    for path in video_paths:
        video_name = os.path.splitext(os.path.basename(path))[0]
        videos[video_name] = path

    return videos


def extract_frame_token(filename_or_base):
    """
    Extrahiert den Frame-Token (z. B. frame_000123) aus einem Dateinamen.
    Funktioniert auch, wenn der Dateiname im manuellen Dataset ein Präfix trägt.
    """
    base = os.path.splitext(os.path.basename(filename_or_base))[0]
    match = re.search(r"(frame_\d+)", base)
    if match:
        return match.group(1)
    return base


def load_forbidden_frame_tokens():
    """
    Sammelt die Frames, die bereits im manuellen Train/Val/Test enthalten sind,
    damit sie nicht ins Pseudo-Dataset gelangen. So wird verhindert:
    - Duplikate im manuellen Training
    - Leakage in die manuelle Validierung
    """
    forbidden = set()

    for split in ["train", "val", "test"]:
        img_dir = os.path.join(MANUAL_DATASET_ROOT, split, "images")

        if not os.path.exists(img_dir):
            continue

        for file in os.listdir(img_dir):
            if not file.lower().endswith((".jpg", ".jpeg", ".png")):
                continue

            token = extract_frame_token(file)
            forbidden.add(token)

    print(f"✅ Manual dataset içinden exclude edilecek frame token sayısı: {len(forbidden)}")
    return forbidden


def get_video_info(video_path):
    # Liest FPS, Auflösung und Frame-Anzahl eines Videos aus
    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        raise RuntimeError(f"Video açılamadı: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    cap.release()

    return fps, width, height, frame_count


def clamp(value, min_value, max_value):
    return max(min_value, min(float(value), max_value))


def xyxy_to_yolo(x1, y1, x2, y2, img_w, img_h):
    """
    Wandelt Pixelkoordinaten (xyxy) ins normierte YOLO-Format um.
    Ungültige oder zu kleine Boxen liefern None.
    """
    x1 = clamp(x1, 0, img_w - 1)
    y1 = clamp(y1, 0, img_h - 1)
    x2 = clamp(x2, 0, img_w - 1)
    y2 = clamp(y2, 0, img_h - 1)

    if x2 <= x1 or y2 <= y1:
        return None

    box_w = x2 - x1
    box_h = y2 - y1

    x_center = (x1 + box_w / 2.0) / img_w
    y_center = (y1 + box_h / 2.0) / img_h
    width = box_w / img_w
    height = box_h / img_h

    area = width * height

    if area < MIN_BOX_AREA_NORM:
        return None

    return x_center, y_center, width, height


def analyze_tracks(track_records):
    """
    Berechnet pro track_id:
    - Länge (Anzahl Frames)
    - mittlere Confidence
    - dominante Klasse
    - Klassenkonsistenz
    und entscheidet, ob der Track behalten wird.
    """
    track_info = {}

    for track_id, records in track_records.items():
        if not records:
            continue

        cls_counter = Counter(r["class_id"] for r in records)
        dominant_class, dominant_count = cls_counter.most_common(1)[0]

        length = len(records)
        avg_conf = sum(r["confidence"] for r in records) / length
        class_consistency = dominant_count / length

        keep = (
            length >= MIN_TRACK_LENGTH
            and avg_conf >= MIN_AVG_TRACK_CONF
            and class_consistency >= MIN_CLASS_CONSISTENCY
        )

        # Ablehnungsgrund für den Bericht festhalten
        reason = "kept"

        if length < MIN_TRACK_LENGTH:
            reason = "too_short"
        elif avg_conf < MIN_AVG_TRACK_CONF:
            reason = "low_avg_conf"
        elif class_consistency < MIN_CLASS_CONSISTENCY:
            reason = "class_inconsistent"

        track_info[track_id] = {
            "length": length,
            "avg_conf": avg_conf,
            "dominant_class": dominant_class,
            "dominant_count": dominant_count,
            "class_consistency": class_consistency,
            "keep": keep,
            "reason": reason,
            "class_counter": dict(cls_counter),
        }

    return track_info


def export_frames_and_labels(video_path, video_name, labels_by_frame, forbidden_tokens):
    """
    Zweiter Durchlauf:
    Öffnet das Video erneut und speichert nur die Frames als Bild,
    für die auch Labels geschrieben werden.
    """
    out_img_dir = os.path.join(OUTPUT_ROOT, video_name, "images")
    out_lbl_dir = os.path.join(OUTPUT_ROOT, video_name, "labels")

    os.makedirs(out_img_dir, exist_ok=True)
    os.makedirs(out_lbl_dir, exist_ok=True)

    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        print(f"❌ Export için video açılamadı: {video_path}")
        return 0, 0, 0

    target_frames = set(labels_by_frame.keys())

    frame_idx = 0
    saved_images = 0
    saved_labels = 0
    skipped_forbidden = 0

    while True:
        ret, frame = cap.read()

        if not ret:
            break

        if frame_idx not in target_frames:
            frame_idx += 1
            continue

        frame_token = f"frame_{frame_idx:06d}"

        # Frames, die im manuellen Train/Val vorkommen, nicht übernehmen
        if frame_token in forbidden_tokens:
            skipped_forbidden += 1
            frame_idx += 1
            continue

        labels = labels_by_frame[frame_idx]

        if not labels:
            frame_idx += 1
            continue

        image_name = f"{frame_token}{OUTPUT_IMAGE_EXT}"
        label_name = f"{frame_token}.txt"

        image_path = os.path.join(out_img_dir, image_name)
        label_path = os.path.join(out_lbl_dir, label_name)

        cv2.imwrite(image_path, frame, [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY])

        with open(label_path, "w", encoding="utf-8") as f:
            for line in labels:
                f.write(line)

        saved_images += 1
        saved_labels += 1

        frame_idx += 1

    cap.release()

    return saved_images, saved_labels, skipped_forbidden


def process_video(model, video_name, video_path, class_names, target_classes, forbidden_tokens):
    print("\n" + "=" * 80)
    print(f"🎬 İşleniyor: {video_name}")
    print("=" * 80)

    fps, img_w, img_h, frame_count = get_video_info(video_path)

    print(f"🎥 FPS: {fps}")
    print(f"🎥 Size: {img_w}x{img_h}")
    print(f"🎥 Frame count: {frame_count}")

    # track_id -> Liste der Detektionen dieses Tracks
    track_records = defaultdict(list)

    total_raw_detections = 0
    total_with_track_id = 0
    total_without_track_id = 0
    target_class_detections = 0

    print("🚀 ByteTrack başlıyor...")

    results = model.track(
        source=video_path,
        tracker=TRACKER_CFG,
        conf=TRACK_CONF_TH,
        imgsz=IMGSZ,
        persist=True,
        stream=True,
        save=False,
        verbose=False,
    )

    frame_idx = 0

    # Erster Durchlauf: alle Detektionen mit Track-ID einsammeln
    for result in results:
        boxes = result.boxes

        if boxes is not None and boxes.xyxy is not None and len(boxes.xyxy) > 0:
            xyxy = boxes.xyxy.cpu().numpy()
            cls_list = boxes.cls.cpu().numpy().astype(int) if boxes.cls is not None else []
            conf_list = boxes.conf.cpu().numpy() if boxes.conf is not None else []

            if boxes.id is not None:
                id_list = boxes.id.cpu().numpy().astype(int)
            else:
                id_list = [-1] * len(xyxy)

            for i in range(len(xyxy)):
                total_raw_detections += 1

                cls_id = int(cls_list[i]) if i < len(cls_list) else -1
                conf = float(conf_list[i]) if i < len(conf_list) else 0.0
                track_id = int(id_list[i]) if i < len(id_list) else -1

                if cls_id not in target_classes:
                    continue

                target_class_detections += 1

                if track_id == -1:
                    total_without_track_id += 1
                    continue

                total_with_track_id += 1

                x1, y1, x2, y2 = xyxy[i]

                track_records[track_id].append({
                    "video_name": video_name,
                    "frame_idx": frame_idx,
                    "track_id": track_id,
                    "class_id": cls_id,
                    "class_name": class_names.get(cls_id, str(cls_id)),
                    "confidence": conf,
                    "x1": float(x1),
                    "y1": float(y1),
                    "x2": float(x2),
                    "y2": float(y2),
                })

        if frame_idx % 250 == 0:
            print(f"🎞️ Frame: {frame_idx}")

        frame_idx += 1

    print("✅ ByteTrack tamamlandı.")
    print(f"📦 Raw detections: {total_raw_detections}")
    print(f"🎯 Target class detections: {target_class_detections}")
    print(f"🆔 Track ID bulunan: {total_with_track_id}")
    print(f"⚠️ Track ID olmayan: {total_without_track_id}")
    print(f"🧵 Unique track: {len(track_records)}")

    # Qualitätsanalyse der Tracks
    track_info = analyze_tracks(track_records)

    kept_tracks = {
        tid: info
        for tid, info in track_info.items()
        if info["keep"]
    }

    removed_tracks = {
        tid: info
        for tid, info in track_info.items()
        if not info["keep"]
    }

    print(f"✅ Kept tracks: {len(kept_tracks)}")
    print(f"❌ Removed tracks: {len(removed_tracks)}")

    removed_reason_counter = Counter(info["reason"] for info in removed_tracks.values())

    if removed_reason_counter:
        print("❌ Removed reasons:")
        for reason, count in removed_reason_counter.items():
            print(f"   - {reason}: {count}")

    # frame_idx -> YOLO-Labelzeilen
    labels_by_frame = defaultdict(list)

    kept_bbox_counter = Counter()
    removed_bbox_counter = Counter()

    detection_rows = []

    for track_id, records in track_records.items():
        info = track_info.get(track_id)

        if info is None or not info["keep"]:
            removed_bbox_counter["track_filtered"] += len(records)
            continue

        dominant_class = info["dominant_class"]

        for r in records:
            # Detektionen außerhalb der dominanten Track-Klasse verwerfen
            if r["class_id"] != dominant_class:
                removed_bbox_counter["non_dominant_class"] += 1
                continue

            # Boxen mit zu niedriger Confidence verwerfen
            if r["confidence"] < KEEP_MIN_CONF:
                removed_bbox_counter["low_conf_bbox"] += 1
                continue

            yolo_box = xyxy_to_yolo(
                r["x1"], r["y1"], r["x2"], r["y2"],
                img_w, img_h
            )

            if yolo_box is None:
                removed_bbox_counter["invalid_box"] += 1
                continue

            x_center, y_center, width, height = yolo_box

            label_line = (
                f"{dominant_class} "
                f"{x_center:.6f} {y_center:.6f} "
                f"{width:.6f} {height:.6f}\n"
            )

            labels_by_frame[r["frame_idx"]].append(label_line)
            kept_bbox_counter[dominant_class] += 1

            detection_rows.append({
                **r,
                "dominant_class": dominant_class,
                "track_length": info["length"],
                "track_avg_conf": info["avg_conf"],
                "class_consistency": info["class_consistency"],
            })

    saved_images, saved_labels, skipped_forbidden = export_frames_and_labels(
        video_path=video_path,
        video_name=video_name,
        labels_by_frame=labels_by_frame,
        forbidden_tokens=forbidden_tokens
    )

    print(f"💾 Saved images: {saved_images}")
    print(f"💾 Saved labels: {saved_labels}")
    print(f"🚫 Skipped manual frames: {skipped_forbidden}")

    print("📊 Kept bbox by class:")
    for cls in sorted(kept_bbox_counter.keys()):
        print(f"{cls:2d} | {class_names.get(cls, cls):20s}: {kept_bbox_counter[cls]}")

    summary = {
        "video_name": video_name,
        "video_path": video_path,
        "fps": fps,
        "width": img_w,
        "height": img_h,
        "frame_count": frame_count,
        "processed_frames": frame_idx,
        "raw_detections": total_raw_detections,
        "target_class_detections": target_class_detections,
        "detections_with_track_id": total_with_track_id,
        "detections_without_track_id": total_without_track_id,
        "unique_tracks": len(track_records),
        "kept_tracks": len(kept_tracks),
        "removed_tracks": len(removed_tracks),
        "saved_images": saved_images,
        "saved_labels": saved_labels,
        "skipped_manual_frames": skipped_forbidden,
        "kept_bbox_total": sum(kept_bbox_counter.values()),
        "removed_bbox_total": sum(removed_bbox_counter.values()),
        "kept_bbox_by_class": dict(kept_bbox_counter),
        "removed_bbox_by_reason": dict(removed_bbox_counter),
        "removed_tracks_by_reason": dict(removed_reason_counter),
    }

    return summary, detection_rows


def save_global_reports(all_summaries, all_detection_rows, class_names):
    # Speichert die globalen Berichte (CSV + JSON) über alle Videos
    os.makedirs(OUTPUT_ROOT, exist_ok=True)

    summary_csv = os.path.join(OUTPUT_ROOT, "tracking_pseudo_summary.csv")
    detections_csv = os.path.join(OUTPUT_ROOT, "tracking_pseudo_detections.csv")
    summary_json = os.path.join(OUTPUT_ROOT, "tracking_pseudo_summary.json")

    # Zusammenfassung als CSV
    with open(summary_csv, "w", newline="", encoding="utf-8") as f:
        fieldnames = [
            "video_name",
            "processed_frames",
            "raw_detections",
            "target_class_detections",
            "detections_with_track_id",
            "detections_without_track_id",
            "unique_tracks",
            "kept_tracks",
            "removed_tracks",
            "saved_images",
            "saved_labels",
            "skipped_manual_frames",
            "kept_bbox_total",
            "removed_bbox_total",
        ]

        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()

        for s in all_summaries:
            writer.writerow({k: s.get(k, "") for k in fieldnames})

    # Detektionen als CSV
    with open(detections_csv, "w", newline="", encoding="utf-8") as f:
        fieldnames = [
            "video_name",
            "frame_idx",
            "track_id",
            "class_id",
            "class_name",
            "confidence",
            "x1", "y1", "x2", "y2",
            "dominant_class",
            "track_length",
            "track_avg_conf",
            "class_consistency",
        ]

        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()

        for row in all_detection_rows:
            writer.writerow({k: row.get(k, "") for k in fieldnames})

    # Einstellungen + Zusammenfassungen als JSON
    with open(summary_json, "w", encoding="utf-8") as f:
        json.dump(
            {
                "settings": {
                    "model_path": MODEL_PATH,
                    "tracker": TRACKER_CFG,
                    "track_conf_th": TRACK_CONF_TH,
                    "imgsz": IMGSZ,
                    "keep_min_conf": KEEP_MIN_CONF,
                    "min_track_length": MIN_TRACK_LENGTH,
                    "min_avg_track_conf": MIN_AVG_TRACK_CONF,
                    "min_class_consistency": MIN_CLASS_CONSISTENCY,
                    "min_box_area_norm": MIN_BOX_AREA_NORM,
                },
                "class_names": {str(k): v for k, v in class_names.items()},
                "summaries": all_summaries,
            },
            f,
            indent=4,
            ensure_ascii=False
        )

    print("\n🔥 Global raporlar kaydedildi:")
    print(f"📄 {summary_csv}")
    print(f"📄 {detections_csv}")
    print(f"📄 {summary_json}")


def main():
    print("🚀 MAIN14 - Tracking-based label propagation başlıyor")

    if not os.path.exists(MODEL_PATH):
        print(f"❌ Baseline model bulunamadı: {MODEL_PATH}")
        print("Önce main5.py ile baseline modelin oluşmuş olması gerekiyor.")
        return

    os.makedirs(OUTPUT_ROOT, exist_ok=True)

    class_names, target_classes = load_target_config()
    forbidden_tokens = load_forbidden_frame_tokens()

    videos = find_videos()

    if not videos:
        print("❌ Video bulunamadı.")
        print(r"Beklenen pattern: C:\thesis\flight-mbg-v2-*\avi\*.avi")
        return

    print("\n🎬 Bulunan videolar:")
    for name, path in videos.items():
        print(f"- {name}: {path}")

    model = YOLO(MODEL_PATH)

    all_summaries = []
    all_detection_rows = []

    for video_name, video_path in videos.items():
        try:
            summary, detection_rows = process_video(
                model=model,
                video_name=video_name,
                video_path=video_path,
                class_names=class_names,
                target_classes=target_classes,
                forbidden_tokens=forbidden_tokens
            )

            all_summaries.append(summary)
            all_detection_rows.extend(detection_rows)

        except Exception as e:
            print(f"❌ Hata oluştu: {video_name}")
            print(str(e))

    save_global_reports(all_summaries, all_detection_rows, class_names)

    print("\n✅ MAIN14 tamamlandı.")
    print(f"Output root: {OUTPUT_ROOT}")


if __name__ == "__main__":
    main()

## main15 — Balanciertes Tracking-Pseudo-Dataset

Wendet dasselbe Quota-Verfahren wie main10 auf die tracking-basierten Pseudo-Labels an und erzeugt ein klassenbalanciertes Trainingsset inklusive JSON-Bericht über die Auswahl.

In [ ]:
# ============================================================
# MAIN15 - BALANCIERTES TRACKING-PSEUDO-DATASET AUFBAUEN
# ============================================================
# Eingabe:
#   C:\thesis\runs\tracking_pseudo\videoXX\images + labels
#
# Ausgabe:
#   C:\thesis\datasets\tracking_pseudo_dataset\train\images
#   C:\thesis\datasets\tracking_pseudo_dataset\train\labels
#
# Zweck:
# Wählt die tracking-basierten Pseudo-Labels (main14) klassen-
# balanciert aus — gleiches Quota-Prinzip wie in main10, aber
# auf den Tracking-Pseudo-Labels statt den Detection-Pseudo-Labels.
# ============================================================

import os
import json
import shutil
import random
from collections import Counter


# =========================
# EINGABE / AUSGABE
# =========================
TRACKING_PSEUDO_ROOT = r"C:\thesis\runs\tracking_pseudo"
OUTPUT_ROOT = r"C:\thesis\datasets\tracking_pseudo_dataset"

CONFIG_PATH = r"C:\thesis\runs\pseudo\target_config.json"

TRAIN_IMG = os.path.join(OUTPUT_ROOT, "train", "images")
TRAIN_LBL = os.path.join(OUTPUT_ROOT, "train", "labels")

IMAGE_EXTS = (".jpg", ".jpeg", ".png")


# =========================
# EINSTELLUNGEN
# =========================
random.seed(42)

# Zielwert pro Klasse: manual + pseudo ≈ dieser Wert.
# Zu hohe Werte lassen dominante Klassen (pool/storm_drain) überwiegen.
TARGET_TOTAL_PER_CLASS = 800

# Maximale Anzahl an Pseudo-Boxen pro Klasse.
MAX_PSEUDO_BBOX_PER_CLASS = 1000

# Obergrenze für die Gesamtzahl der ausgewählten Pseudo-Frames.
MAX_TOTAL_FRAMES = 3000


def load_config():
    # Lädt Klassen, Zielklassen und manuelle Verteilung aus target_config.json
    if not os.path.exists(CONFIG_PATH):
        raise FileNotFoundError(
            f"Config bulunamadı: {CONFIG_PATH}. Önce main6.py ile target_config oluşmuş olmalı."
        )

    with open(CONFIG_PATH, "r", encoding="utf-8") as f:
        config = json.load(f)

    class_names = {int(k): v for k, v in config["class_names"].items()}
    target_classes = [int(x) for x in config["target_classes"]]

    manual_distribution = {
        int(k): int(v)
        for k, v in config.get("manual_distribution", {}).items()
    }

    return class_names, target_classes, manual_distribution


def clean_output():
    # Alte Ausgabe löschen und leere Ordnerstruktur anlegen
    if os.path.exists(OUTPUT_ROOT):
        shutil.rmtree(OUTPUT_ROOT)

    os.makedirs(TRAIN_IMG, exist_ok=True)
    os.makedirs(TRAIN_LBL, exist_ok=True)


def read_label_lines(label_path):
    """
    YOLO-Labelformat:
    class x_center y_center width height
    """
    lines = []

    with open(label_path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()

            if len(parts) < 5:
                continue

            cls = int(parts[0])
            clean_line = " ".join(parts[:5]) + "\n"
            lines.append((cls, clean_line))

    return lines


def find_all_tracking_pairs():
    """
    Findet in tracking_pseudo die Paare:
    video01_regionA/images/frame_xxx.jpg
    video01_regionA/labels/frame_xxx.txt
    """
    pairs = []

    video_dirs = [
        d for d in sorted(os.listdir(TRACKING_PSEUDO_ROOT))
        if os.path.isdir(os.path.join(TRACKING_PSEUDO_ROOT, d))
    ]

    for video_name in video_dirs:
        img_dir = os.path.join(TRACKING_PSEUDO_ROOT, video_name, "images")
        lbl_dir = os.path.join(TRACKING_PSEUDO_ROOT, video_name, "labels")

        if not os.path.exists(img_dir) or not os.path.exists(lbl_dir):
            continue

        images = [
            f for f in sorted(os.listdir(img_dir))
            if f.lower().endswith(IMAGE_EXTS)
        ]

        video_count = 0

        for img_file in images:
            base, ext = os.path.splitext(img_file)
            lbl_file = base + ".txt"

            src_img = os.path.join(img_dir, img_file)
            src_lbl = os.path.join(lbl_dir, lbl_file)

            if not os.path.exists(src_lbl):
                continue

            pairs.append({
                "video_name": video_name,
                "src_img": src_img,
                "src_lbl": src_lbl,
                "img_file": img_file,
                "base": base,
                "ext": ext,
            })

            video_count += 1

        print(f"📹 {video_name}: {video_count} tracking pseudo image-label bulundu")

    return pairs


def calculate_quota(class_names, target_classes, manual_distribution):
    # Pseudo-Quota pro Klasse: Zielwert minus manuelle Boxen, gedeckelt
    quota = {}

    print("\n📊 Manual dağılıma göre tracking pseudo quota:")

    for cls in sorted(target_classes):
        manual_count = manual_distribution.get(cls, 0)

        missing = TARGET_TOTAL_PER_CLASS - manual_count
        pseudo_quota = max(0, min(missing, MAX_PSEUDO_BBOX_PER_CLASS))

        quota[cls] = pseudo_quota

        print(
            f"{cls:2d} | {class_names.get(cls, cls):20s} "
            f"| manual: {manual_count:5d} "
            f"| pseudo quota: {pseudo_quota:5d}"
        )

    return quota


def main():
    print("🚀 MAIN15 - Balanced tracking pseudo dataset oluşturuluyor")

    class_names, target_classes, manual_distribution = load_config()
    target_classes = set(target_classes)

    clean_output()

    print("\n🎯 Target classes:")
    for cls in sorted(target_classes):
        print(f"{cls:2d} | {class_names.get(cls, cls)}")

    quota = calculate_quota(class_names, target_classes, manual_distribution)

    if sum(quota.values()) == 0:
        print("\n⚠️ Hiç pseudo quota yok. Output oluşturulmadı.")
        return

    all_pairs = find_all_tracking_pairs()
    random.shuffle(all_pairs)

    print(f"\n📦 Toplam aday tracking pseudo frame: {len(all_pairs)}")

    # Frames mit vielen Boxen aus unterrepräsentierten Klassen priorisieren
    def priority_score(item):
        label_lines = read_label_lines(item["src_lbl"])
        score = 0

        for cls, _line in label_lines:
            if cls in quota and quota[cls] > 0:
                score += quota[cls]

        return score

    all_pairs.sort(key=priority_score, reverse=True)

    used_bbox_counter = Counter()
    selected_frames = 0
    skipped_frames = 0
    total_seen_bbox = Counter()

    for item in all_pairs:
        if selected_frames >= MAX_TOTAL_FRAMES:
            break

        label_lines = read_label_lines(item["src_lbl"])

        kept_lines = []

        for cls, line in label_lines:
            total_seen_bbox[cls] += 1

            if cls not in target_classes:
                continue

            # Klassen mit erfüllter Quota nicht weiter aufnehmen
            if used_bbox_counter[cls] >= quota.get(cls, 0):
                continue

            kept_lines.append(line)
            used_bbox_counter[cls] += 1

        if not kept_lines:
            skipped_frames += 1
            continue

        video_name = item["video_name"]
        base = item["base"]
        ext = item["ext"]

        # Videoname als Präfix gegen Namenskollisionen
        new_img_name = f"{video_name}_{base}{ext}"
        new_lbl_name = f"{video_name}_{base}.txt"

        shutil.copy2(item["src_img"], os.path.join(TRAIN_IMG, new_img_name))

        with open(os.path.join(TRAIN_LBL, new_lbl_name), "w", encoding="utf-8") as f:
            f.writelines(kept_lines)

        selected_frames += 1

        # Abbruch, sobald alle Quotas erfüllt sind
        all_full = all(
            used_bbox_counter[cls] >= quota.get(cls, 0)
            for cls in target_classes
        )

        if all_full:
            break

    print("\n✅ Balanced tracking pseudo train dataset oluşturuldu")
    print(f"📄 Seçilen frame: {selected_frames}")
    print(f"⏭️ Boş/işe yaramayan frame: {skipped_frames}")

    print("\n📊 Seçilen tracking pseudo bbox dağılımı:")
    for cls in sorted(target_classes):
        print(
            f"{cls:2d} | {class_names.get(cls, cls):20s}: "
            f"{used_bbox_counter[cls]:5d} / quota {quota[cls]:5d}"
        )

    print("\n📊 Adaylarda görülen toplam bbox:")
    for cls in sorted(total_seen_bbox.keys()):
        print(
            f"{cls:2d} | {class_names.get(cls, cls):20s}: "
            f"{total_seen_bbox[cls]}"
        )

    # Auswahlbericht als JSON speichern
    report_path = os.path.join(OUTPUT_ROOT, "tracking_pseudo_dataset_report.json")

    with open(report_path, "w", encoding="utf-8") as f:
        json.dump(
            {
                "settings": {
                    "target_total_per_class": TARGET_TOTAL_PER_CLASS,
                    "max_pseudo_bbox_per_class": MAX_PSEUDO_BBOX_PER_CLASS,
                    "max_total_frames": MAX_TOTAL_FRAMES,
                },
                "selected_frames": selected_frames,
                "skipped_frames": skipped_frames,
                "quota": {str(k): v for k, v in quota.items()},
                "selected_bbox": {str(k): v for k, v in used_bbox_counter.items()},
                "seen_bbox": {str(k): v for k, v in total_seen_bbox.items()},
                "class_names": {str(k): v for k, v in class_names.items()},
            },
            f,
            indent=4,
            ensure_ascii=False
        )

    print(f"\n🔥 Output: {OUTPUT_ROOT}")
    print(f"📄 Report: {report_path}")


if __name__ == "__main__":
    main()

## main16 — Tracking-Semi-Supervised-Dataset & data.yaml

Führt manuelles Training + Tracking-Pseudo-Labels zusammen (Validierung wieder nur manuell), prüft die Image/Label-Paare, gibt die Klassenverteilung aus und erzeugt automatisch die `data_tracking_semi.yaml` für das YOLO-Training.

In [ ]:
# ============================================================
# MAIN16 - TRACKING-SEMI-SUPERVISED-DATASET AUFBAUEN
# ============================================================
# Eingabe:
#   Manuelles Dataset:
#       C:\thesis\datasets\merged_dataset
#
#   Tracking-Pseudo-Dataset:
#       C:\thesis\datasets\tracking_pseudo_dataset
#
# Ausgabe:
#   C:\thesis\datasets\tracking_semi_supervised_dataset
#
# Train:
#   manuelles Training + Tracking-Pseudo-Training
#
# Val:
#   ausschließlich manuelle Validierung
#
# Zusätzlich werden die Image/Label-Paare geprüft, die Klassen-
# verteilung ausgegeben und die data_tracking_semi.yaml erzeugt.
# ============================================================

import os
import shutil
import yaml
from collections import Counter


# =========================
# PFADE
# =========================
MANUAL_ROOT = r"C:\thesis\datasets\merged_dataset"
TRACKING_PSEUDO_ROOT = r"C:\thesis\datasets\tracking_pseudo_dataset"

OUTPUT_ROOT = r"C:\thesis\datasets\tracking_semi_supervised_dataset"

DATA_YAML_PATH = r"C:\thesis\data_tracking_semi.yaml"

IMAGE_EXTS = (".jpg", ".jpeg", ".png")


CLASS_NAMES = {
    0: "pool",
    1: "bottle",
    2: "bucket",
    3: "puddle",
    4: "tire",
    5: "watertank",
    6: "dumpster",
    7: "large_trash_bin",
    8: "plastic_bag",
    9: "small_trash_bin",
    10: "storm_drain",
}


def clean_output():
    # Alte Ausgabe löschen und leere Ordnerstruktur anlegen
    if os.path.exists(OUTPUT_ROOT):
        shutil.rmtree(OUTPUT_ROOT)

    for split in ["train", "val"]:
        os.makedirs(os.path.join(OUTPUT_ROOT, split, "images"), exist_ok=True)
        os.makedirs(os.path.join(OUTPUT_ROOT, split, "labels"), exist_ok=True)


def copy_split(src_root, split, dst_split, prefix):
    # Kopiert einen Split mit Herkunfts-Präfix ("manual"/"trackingpseudo")
    src_img_dir = os.path.join(src_root, split, "images")
    src_lbl_dir = os.path.join(src_root, split, "labels")

    dst_img_dir = os.path.join(OUTPUT_ROOT, dst_split, "images")
    dst_lbl_dir = os.path.join(OUTPUT_ROOT, dst_split, "labels")

    if not os.path.exists(src_img_dir):
        print(f"❌ Image klasörü yok: {src_img_dir}")
        return 0

    if not os.path.exists(src_lbl_dir):
        print(f"❌ Label klasörü yok: {src_lbl_dir}")
        return 0

    count = 0

    for img_file in sorted(os.listdir(src_img_dir)):
        if not img_file.lower().endswith(IMAGE_EXTS):
            continue

        base, ext = os.path.splitext(img_file)
        lbl_file = base + ".txt"

        src_img = os.path.join(src_img_dir, img_file)
        src_lbl = os.path.join(src_lbl_dir, lbl_file)

        # Nur vollständige Bild/Label-Paare übernehmen
        if not os.path.exists(src_lbl):
            continue

        new_img_name = f"{prefix}_{base}{ext}"
        new_lbl_name = f"{prefix}_{base}.txt"

        shutil.copy2(src_img, os.path.join(dst_img_dir, new_img_name))
        shutil.copy2(src_lbl, os.path.join(dst_lbl_dir, new_lbl_name))

        count += 1

    print(f"✅ {prefix}: {split} → {dst_split}: {count} image-label kopyalandı")
    return count


def count_labels(label_dir):
    # Zählt Labeldateien und Boxen pro Klasse
    counter = Counter()
    image_label_files = 0

    if not os.path.exists(label_dir):
        return counter, image_label_files

    for file in os.listdir(label_dir):
        if not file.endswith(".txt"):
            continue

        image_label_files += 1

        with open(os.path.join(label_dir, file), "r", encoding="utf-8") as f:
            for line in f:
                parts = line.strip().split()

                if len(parts) < 5:
                    continue

                cls = int(parts[0])
                counter[cls] += 1

    return counter, image_label_files


def check_image_label_pairs(split):
    # Prüft, ob jedes Bild ein Label hat und umgekehrt
    img_dir = os.path.join(OUTPUT_ROOT, split, "images")
    lbl_dir = os.path.join(OUTPUT_ROOT, split, "labels")

    images = {
        os.path.splitext(f)[0]
        for f in os.listdir(img_dir)
        if f.lower().endswith(IMAGE_EXTS)
    }

    labels = {
        os.path.splitext(f)[0]
        for f in os.listdir(lbl_dir)
        if f.endswith(".txt")
    }

    missing_labels = images - labels
    extra_labels = labels - images

    print(f"\n🔍 Pair kontrolü - {split}")
    print(f"Images: {len(images)}")
    print(f"Labels: {len(labels)}")
    print(f"Label olmayan image: {len(missing_labels)}")
    print(f"Image olmayan label: {len(extra_labels)}")

    return len(missing_labels), len(extra_labels)


def create_data_yaml():
    # Erzeugt die YOLO-Datenkonfiguration für das Training (main17)
    data = {
        "path": OUTPUT_ROOT.replace("\\", "/"),
        "train": "train/images",
        "val": "val/images",
        "names": CLASS_NAMES,
    }

    with open(DATA_YAML_PATH, "w", encoding="utf-8") as f:
        yaml.dump(data, f, sort_keys=False, allow_unicode=True)

    print(f"\n✅ data_tracking_semi.yaml oluşturuldu:")
    print(DATA_YAML_PATH)


def print_distribution(split):
    # Gibt die Klassenverteilung eines Splits aus
    label_dir = os.path.join(OUTPUT_ROOT, split, "labels")
    counter, file_count = count_labels(label_dir)

    print(f"\n📊 {split.upper()} label dağılımı")
    print(f"Label dosyası: {file_count}")

    total = sum(counter.values())
    print(f"Toplam bbox: {total}")

    for cls in sorted(CLASS_NAMES.keys()):
        print(f"{cls:2d} | {CLASS_NAMES[cls]:20s}: {counter.get(cls, 0)}")


def main():
    print("🚀 MAIN16 - Tracking semi-supervised dataset oluşturuluyor")

    clean_output()

    # 1) Manuelles Training -> train
    manual_train = copy_split(
        src_root=MANUAL_ROOT,
        split="train",
        dst_split="train",
        prefix="manual"
    )

    # 2) Tracking-Pseudo-Training -> train
    tracking_pseudo_train = copy_split(
        src_root=TRACKING_PSEUDO_ROOT,
        split="train",
        dst_split="train",
        prefix="trackingpseudo"
    )

    # 3) Manuelle Validierung -> val (keine Pseudo-Labels in der Validierung!)
    manual_val = copy_split(
        src_root=MANUAL_ROOT,
        split="val",
        dst_split="val",
        prefix="manual"
    )

    print("\n🔥 Tracking semi-supervised dataset hazır:")
    print(OUTPUT_ROOT)

    print("\n📊 Özet:")
    print(f"Train toplam: {manual_train + tracking_pseudo_train}")
    print(f"  - Manual train: {manual_train}")
    print(f"  - Tracking pseudo train: {tracking_pseudo_train}")
    print(f"Val toplam: {manual_val}")
    print(f"  - Manual val only: {manual_val}")

    check_image_label_pairs("train")
    check_image_label_pairs("val")

    print_distribution("train")
    print_distribution("val")

    create_data_yaml()

    print("\n✅ MAIN16 tamamlandı.")


if __name__ == "__main__":
    main()

## main17 — Training mit Tracking-Label-Propagation

Fine-Tuning des Baseline-Modells mit dem Tracking-Semi-Supervised-Dataset. Sehr kleine Lernrate (`lr0=0.00015`) für vorsichtiges Anpassen. Ergebnis: Modell `tracking_label_propagation_iter2_patience20`.

In [ ]:
# ============================================================
# MAIN17 - TRAINING MIT TRACKING-BASIERTER LABEL-PROPAGATION
# ============================================================
# Zweck:
# Fine-Tuning ausgehend vom Baseline-Modell mit dem Dataset aus
# manuellem Training + Tracking-Pseudo-Labels (main16).
#
# Validierung:
# Es wird ausschließlich die manuelle Validierung verwendet.
# ============================================================

from ultralytics import YOLO


MODEL_PATH = r"C:\thesis\runs\detect\train\weights\best.pt"
DATA_YAML = r"C:\thesis\data_tracking_semi.yaml"


def main():
    # Startpunkt: die besten Gewichte des Baseline-Trainings
    model = YOLO(MODEL_PATH)

    model.train(
        data=DATA_YAML,
        epochs=60,
        imgsz=1024,
        device=0,
        patience=20,
        batch=16,
        workers=4,
        optimizer="AdamW",
        lr0=0.00015,        # sehr kleine Lernrate für vorsichtiges Fine-Tuning
        cos_lr=True,
        augment=True,
        project=r"C:\thesis\runs\detect",
        name="tracking_label_propagation_iter2_patience20"
    )


if __name__ == "__main__":
    main()

# Phase 6 — Evaluation & Modellvergleich (main18)

## main18 — Modellvergleich auf identischem Validierungsset

Abschließende Evaluation: Baseline, Detection-Pseudo- und Tracking-Pseudo-Modell werden auf **demselben manuellen Validierungsset** validiert. Precision, Recall, mAP50 und mAP50-95 werden als Vergleichs-CSV gespeichert und als Tabelle ausgegeben.

In [ ]:
# ============================================================
# MAIN18 - VERGLEICH: BASELINE / DETECTION-PSEUDO / TRACKING-PSEUDO
# ============================================================
# Zweck:
# Validiert alle drei Modelle (Baseline, Detection-Pseudo,
# Tracking-Pseudo) auf demselben manuellen Validierungsset und
# schreibt die Metriken (Precision, Recall, mAP50, mAP50-95)
# in eine Vergleichs-CSV.
#
# Wichtig:
# Es wird nur die Validierung genutzt. Da val in
# data_tracking_semi.yaml = manuelle Validierung ist, werden
# alle Modelle auf identischen Daten fair verglichen.
# ============================================================

import os
import csv
from ultralytics import YOLO

DATA_YAML = r"C:\thesis\data_tracking_semi.yaml"

OUTPUT_CSV = r"C:\thesis\runs\detect\model_comparison_same_val.csv"

# Die drei zu vergleichenden Modelle mit ihren Trainingsdaten
MODELS = [
    {
        "name": "baseline_manual_only",
        "training_data": "manual train only",
        "path": r"C:\thesis\runs\detect\train\weights\best.pt",
    },
    {
        "name": "detection_pseudo_iter1",
        "training_data": "manual train + detection pseudo-labels",
        "path": r"C:\thesis\runs\detect\semi_iter1_targeted\weights\best.pt",
    },
    {
        "name": "tracking_label_propagation_iter2_patience20",
        "training_data": "manual train + ByteTrack tracking pseudo-labels",
        "path": r"C:\thesis\runs\detect\tracking_label_propagation_iter2_patience20\weights\best.pt",
    },
]


def main():
    rows = []

    for item in MODELS:
        model_path = item["path"]

        # Fehlende Modelle überspringen statt abzubrechen
        if not os.path.exists(model_path):
            print(f"⚠️ Model bulunamadı, atlanıyor: {item['name']}")
            print(model_path)
            continue

        print("\n" + "=" * 100)
        print(f"🔍 Validating: {item['name']}")
        print("=" * 100)

        model = YOLO(model_path)

        # Validierung auf dem gemeinsamen manuellen Val-Set
        results = model.val(
            data=DATA_YAML,
            imgsz=1024,
            device=0,
            batch=16,
            split="val",
            project=r"C:\thesis\runs\detect\comparison_val",
            name=item["name"],
            verbose=True,
            plots=True
        )

        row = {
            "model": item["name"],
            "training_data": item["training_data"],
            "precision": float(results.box.mp),
            "recall": float(results.box.mr),
            "map50": float(results.box.map50),
            "map50_95": float(results.box.map),
        }

        rows.append(row)

        print("\n📊 Result:")
        print(row)

    if not rows:
        print("❌ Hiç model validate edilemedi.")
        return

    os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)

    # Ergebnisse als CSV speichern
    with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
        fieldnames = [
            "model",
            "training_data",
            "precision",
            "recall",
            "map50",
            "map50_95"
        ]

        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

    print("\n🔥 Comparison CSV kaydedildi:")
    print(OUTPUT_CSV)

    # Abschließende Vergleichstabelle in der Konsole ausgeben
    print("\n📊 FINAL COMPARISON")
    print("=" * 110)
    print(
        f"{'Model':45s} "
        f"{'Precision':>10s} "
        f"{'Recall':>10s} "
        f"{'mAP50':>10s} "
        f"{'mAP50-95':>10s}"
    )
    print("-" * 110)

    for r in rows:
        print(
            f"{r['model']:45s} "
            f"{r['precision']:10.3f} "
            f"{r['recall']:10.3f} "
            f"{r['map50']:10.3f} "
            f"{r['map50_95']:10.3f}"
        )


if __name__ == "__main__":
    main()